# Ozone and temperature in 2025

We examine sustained periods of elevated ozone at McAdory and the temperatures observed at the nearby Alabaster weather station in 2025.

Bronze preserves the source downloads and stores raw tables as Parquet. Silver cleans the records and matches them by UTC hour. Gold produces a catalogue of ozone periods and monthly coverage to help interpret the results.

We use [EPA AirData](https://aqs.epa.gov/aqsweb/airdata/FileFormats.html) for ozone and [NOAA GHCNh](https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh_DOCUMENTATION.pdf) for temperature. The analysis describes this pairing and the available observations. It does not establish a causal relationship or assess regulatory compliance.


## 1. Setup

Load the settings and select the runtime and storage environment. VM runs use the `spark-env` kernel. Local verification uses `dat535-local` and separate test storage. Both use Spark/PySpark 3.5.0.

Full runs every layer, Prepare stops after hourly Silver, and Analyse runs Gold from a selected Silver completion report. Running the notebook top to bottom follows Full. Set `PIPELINE_SHOW_DIAGNOSTICS=true` for extra sample rows.

Development, test and production use separate data roots. GitHub Actions is configured to check pull requests and run Full on pushes to `dev` or `main`.

Spark transformations describe the work. Actions such as `count`, `show` and Parquet writes execute it.


In [1]:
# Load configuration and run helpers before importing PySpark.
import hashlib
import json
import os
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from functools import partial
from pathlib import Path

# Supported starting folders: the checkout, final_project, or notebooks.
folder = Path.cwd().resolve()
if folder.name == "notebooks":
    project_dir = folder.parent
elif folder.name == "final_project":
    project_dir = folder
else:
    project_dir = folder / "final_project"

sys.path[:0] = [str(project_dir / "src"), str(project_dir / "tests")]
from project_config import load_run_settings
from run_context import RunContext, capture_provenance, create_run_context
from pipeline_files import (
    child_path,
    file_sha256,
    json_sha256,
    verify_file_inventory,
    write_json,
)
from spark_runtime import configure_runtime, start_session

settings = load_run_settings(project_dir, os.environ)
environments_file = Path(settings["environments_file"])
show_diagnostics = settings["show_diagnostics"]


### Start Spark and record the run

One run ID links the Bronze, Silver and Gold reports. Print the report folder before starting Spark in UTC, then check the Spark/PySpark versions and selected master.

The reports record the settings used for the run, Git revision, runtime versions and whether there are uncommitted changes.


In [2]:
context = create_run_context(
    environments_file,
    settings["environment"],
    test_run_id=settings["test_run_id"],
)

# Print the output location before runtime-dependent operations can fail.
context.reports_dir.mkdir(parents=True, exist_ok=True)
print("Reports directory:", context.reports_dir, flush=True)
configure_runtime(settings["runtime"])
import pyspark
from pyspark.sql import Window, functions as F
# Import the Silver loaders and shared validation helpers.
from silver_io import (
    check_hourly_evidence,
    hourly_expected_rows,
    hourly_output_keys,
    load_hourly_silver,
    load_source_silver,
    read_hourly_files,
)
# Use one UTC Spark session for Bronze, Silver and Gold.
spark = start_session(project_dir, context, runtime=settings["runtime"])
print("Spark engine:", spark.version)
print("Master:", spark.sparkContext.master)
print("Session timezone:", spark.conf.get("spark.sql.session.timeZone"))
print("Environment root:", context.environment_root)
settings["environment_root"] = str(context.environment_root)
settings["runtime_details"] = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "master": spark.sparkContext.master,
    "session_timezone": spark.conf.get("spark.sql.session.timeZone"),
}
run_provenance = capture_provenance(context, project_dir, settings)


Reports directory: /home/ubuntu/dat535-final-project-data/dev/runs/0986e41d03894eabbfdfdacf001f3362/reports


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 16:12:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark engine: 3.5.0
Master: spark://sparkvm-test:7077
Session timezone: UTC
Environment root: /home/ubuntu/dat535-final-project-data/dev


## 2. Alabama profile

Load the monitor, weather station and year from `alabama_profile.json`. The profile's provisional status is separate from the hourly acceptance decision recorded in section 8.


In [3]:
# Load the Alabama monitor, weather station and year used by the main pipeline.
alabama_profile = json.loads(
    (project_dir / "config" / "alabama_profile.json").read_text(encoding="utf-8")
)


## 3. Bronze: accepted downloads and Spark Parquet

We analyse 2025 observations alongside fixed EPA and NOAA reference snapshots. The national ozone input contains about 8.87 million rows. Bronze converts the full source to Parquet before Silver selects the McAdory monitor.

The five checksums in `sources.json` identify the accepted ZIP/CSV/PSV downloads. Bronze preserves the originals and writes their raw fields as strings in Parquet. Missing required columns stop conversion.

On later runs, Bronze verifies and reuses completed Parquet versions. A missing version is built from a verified cached download or a newly downloaded file matching the checksum. ZIP members are extracted temporarily during conversion.


In [4]:
from bronze import (
    attach_bronze_lineage,
    prepare_bronze,
    read_source,
    selected_bronze_lineage,
)

selected_sources, raw_tables = prepare_bronze(
    spark,
    settings["configured_sources"],
    context,
    run_provenance,
)


26/11/20 16:12:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


epa/sites: created (20994 rows)


epa/monitors: created (370090 rows)


epa/hourly_ozone_2025: created (8867266 rows)
noaa/stations: created (38870 rows)


noaa/hourly_weather_2025_USW00053864: created (11851 rows)
Bronze report: /home/ubuntu/dat535-final-project-data/dev/runs/0986e41d03894eabbfdfdacf001f3362/reports/run.json


The next cell lists the five Bronze DataFrames and their string-column counts.

When Silver adds source details to the rows, `source_file_path` names the preserved ZIP/CSV/PSV. `source_file_sha256` and `source_version_sha256` contain its accepted checksum. These details let us trace a row back to its download. Parquet files have their own checksums.


In [5]:
for key, records in raw_tables.items():
    print(f"{key}: {len(records.columns)} string columns")

print(f"Bronze ready: {len(raw_tables)} sources.")


epa/sites: 28 string columns
epa/monitors: 32 string columns
epa/hourly_ozone_2025: 24 string columns
noaa/stations: 11 string columns
noaa/hourly_weather_2025_USW00053864: 329 string columns
Bronze ready: 5 sources.


## 4. Silver: ozone observations

### 4.1 Shared field checks

`missing_string` identifies null or blank text. `finite_number` excludes null, NaN and infinite values. The source-cleaning functions below reuse both checks.


In [6]:
def missing_string(column_name):
    """Identify null, empty or space-only source values."""
    value = F.col(column_name)
    return value.isNull() | (F.trim(value) == "")


def finite_number(column_name):
    """Return a Spark condition excluding null, NaN and positive or negative infinity."""
    value = F.col(column_name)
    return (
        value.isNotNull()
        & ~F.isnan(value)
        & (F.abs(value) != float("inf"))
    )


### 4.2 Apply the ozone quality rules

Combine the EPA date and time into a UTC timestamp, and convert ozone and method detection limit (MDL) values to numbers. The numeric conversions use Spark SQL expressions such as `try_cast`. Text that cannot be converted becomes null. We keep the original text so the quality checks and audit can show what failed.

Check identifiers, timestamps, units, measurements and qualifiers. Each row keeps all failure reasons, with the first used as its summary outcome.

Qualifier `6` and unsupported qualifier codes exclude a reading. Small negative ozone values down to -0.004 ppm are retained and flagged. A missing or unusable MDL does not exclude an otherwise usable reading. These functions classify the rows; the observation and audit tables are built later.


In [7]:
# Convert dates and measurements from text, keeping the original columns.
# Values that cannot be parsed become null for the later checks.
def type_ozone_fields(records):
    """Add typed fields while retaining the source columns."""

    # Combine the EPA GMT date and time, then parse them using the UTC Spark session.
    sample_time = F.concat(
        F.col("Date GMT"), F.lit(" "), F.col("Time GMT")
    )

    return (
        records
        .withColumn(
            "timestamp_utc",
            F.try_to_timestamp(sample_time, F.lit("yyyy-MM-dd HH:mm")),
        )
        .withColumn(
            "ozone_value",
            F.expr("try_cast(`Sample Measurement` AS DOUBLE)"),
        )
        .withColumn(
            "mdl_value",
            F.expr("try_cast(`MDL` AS DOUBLE)"),
        )
    )

# Check for missing required identifiers: state, county, site, parameter, POC. 
# These Spark conditions describe checks; the processing function applies them to rows.
missing_identifier = F.lit(False)
for column_name in (
    "State Code", "County Code", "Site Num", "Parameter Code", "POC"
):
    missing_identifier = missing_identifier | missing_string(column_name)

# Check for missing date, time, or measurement units.
missing_time = missing_string("Date GMT") | missing_string("Time GMT")
missing_units = missing_string("Units of Measure")

ozone_field_checks = {
    "missing_identifier": missing_identifier,
    "missing_timestamp": missing_time,
    "invalid_timestamp": ~missing_time & F.col("timestamp_utc").isNull(),
    "missing_units": missing_units,
    "unsupported_units": (
        ~missing_units
        & (F.col("Units of Measure") != "Parts per million")
    ),
}

# Check ozone and MDL for missing, invalid, non-finite, or negative values.
for label, source_column, typed_column in (
    ("ozone", "Sample Measurement", "ozone_value"),
    ("mdl", "MDL", "mdl_value"),
):
    # Check for missing text and a finite numeric value.
    missing = missing_string(source_column)
    value = F.col(typed_column)
    finite = finite_number(typed_column)

    ozone_field_checks.update({
        f"missing_{label}": missing,
        f"invalid_{label}": ~missing & value.isNull(),
        f"nonfinite_{label}": value.isNotNull() & ~finite,
        f"negative_{label}": finite & (value < 0),
    })


# Mark which rows pass the quality rules and keep the reasons for failures.
def classify_ozone_fields(records, year):
    """Classify checked ozone records using ordered field rules."""
    # Normalise qualifier text before comparing it with the supported codes.
    allowed_qualifiers = ["", "XS", "MD", "1V", "V"]
    qualifier = F.coalesce(F.trim(F.col("Qualifier")), F.lit(""))
    timestamp = F.col("timestamp_utc")
    value = F.col("ozone_value")

    # Keep all failures, but use the first in this list as the single audit outcome.
    rules = [
        ("missing_identifier", F.col("missing_identifier")),
        ("unusable_timestamp", timestamp.isNull()),
        (
            "outside_year",
            timestamp.isNotNull() & (F.year(timestamp) != year),
        ),
        (
            "off_hour",
            timestamp.isNotNull()
            & (timestamp != F.date_trunc("hour", timestamp)),
        ),
        (
            "unusable_units",
            missing_string("Units of Measure")
            | (F.col("Units of Measure") != "Parts per million"),
        ),
        ("unusable_ozone", ~finite_number("ozone_value")),
        (
            "below_minimum",
            finite_number("ozone_value") & (value < -0.004),
        ),
        ("qualifier_6", qualifier == "6"),
        (
            "unsupported_qualifier",
            ~qualifier.isin(allowed_qualifiers + ["6"]),
        ),
    ]

    # Collect all failure reasons in rule order, leaving out rules that passed.
    reasons = F.filter(
        F.array(*[
            F.when(condition, F.lit(reason))
            for reason, condition in rules
        ]),
        lambda reason: reason.isNotNull(),
    )

    return (
        records
        .withColumn("qualifier_code", qualifier)
        .withColumn("field_reasons", reasons)
        .withColumn("field_eligible", F.size("field_reasons") == 0)
        .withColumn(
            "field_disposition",
            F.when(F.col("field_eligible"), F.lit("eligible"))
            .otherwise(F.element_at("field_reasons", 1)),
        )
    )


### 4.3 Resolve duplicate readings

We expect one ozone observation per monitor and UTC timestamp.

The function groups exact copies and counts them. Different records with the same usable key in the same source file and version are excluded from the clean observations and kept in the audit with a conflict flag.

These decisions and copy counts are used to build the observation and audit tables.


In [8]:
# Together these identifiers and the UTC timestamp identify one ozone observation.
ozone_key_columns = [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC",
    "timestamp_utc",
]

def resolve_source_keys(records, key_columns, key_is_usable):
    """Count identical copies and mark different rows sharing a usable key as ineligible."""

    # Keep conflict checks within the same source file and version.
    source_scope = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
    ]

    # Group rows that match in every column and count their copies.
    distinct_records = records.groupBy(*records.columns).agg(
        F.count("*").alias("source_occurrences")
    )

    # The window groups matching source/key rows without collapsing them.
    # After exact copies are grouped, more than one row means conflicting evidence.
    key_window = Window.partitionBy(*(source_scope + key_columns))
    usable_key = F.coalesce(key_is_usable, F.lit(False))

    return (
        distinct_records
        .withColumn(
            "duplicate_extra_records",
            F.col("source_occurrences") - 1,
        )
        .withColumn(
            "key_variant_count",
            F.when(
                usable_key,
                F.count(F.lit(1)).over(key_window),
            ),
        )
        .withColumn(
            "key_conflict",
            F.coalesce(F.col("key_variant_count") > 1, F.lit(False)),
        )
        .withColumn(
            "record_disposition",
            F.when(F.col("key_conflict"), F.lit("conflicting_key"))
            .otherwise(F.col("field_disposition")),
        )
        .withColumn(
            "record_reasons",
            F.when(
                F.col("key_conflict"),
                F.concat(
                    F.array(F.lit("conflicting_key")),
                    F.col("field_reasons"),
                ),
            ).otherwise(F.col("field_reasons")),
        )
        .withColumn(
            "record_eligible",
            F.col("record_disposition") == "eligible",
        )
    )


### 4.4 Build the ozone observation table

Keep eligible readings with their monitor ID, UTC timestamp and ozone value. Retain the source text and Bronze lineage, meaning the source details used to trace an observation to its input.

Store the MDL when it is finite and non-negative; otherwise use null. Quality flags describe retained readings, including negative values, readings below the MDL and source qualifiers.


In [9]:
# Keep eligible ozone readings with quality flags and source details.
def build_ozone_observations(records):
    """Build the eligible ozone observation schema."""

    identifier_columns = {
        "State Code": "state_code",
        "County Code": "county_code",
        "Site Num": "site_number",
        "Parameter Code": "parameter_code",
        "POC": "poc",
    }

    # Keep the detection limit only if it is finite and non-negative.
    # An unusable detection limit does not remove an otherwise eligible ozone reading.
    mdl_usable = (
        finite_number("mdl_value")
        & (F.col("mdl_value") >= 0)
    )

    # Describe retained readings with flags; these flags do not exclude the reading.
    flag_rules = {
        "negative_ozone": F.col("negative_ozone"),
        "below_reported_mdl": (
            mdl_usable & (F.col("ozone_value") < F.col("mdl_value"))
        ),
        "source_qualifier_XS": F.col("qualifier_code") == "XS",
        "source_qualifier_MD": F.col("qualifier_code") == "MD",
        "source_validation_qualifier": (
            F.col("qualifier_code").isin("1V", "V")
        ),
        "missing_mdl": F.col("missing_mdl"),
        "invalid_mdl": F.col("invalid_mdl"),
        "nonfinite_mdl": F.col("nonfinite_mdl"),
        "negative_mdl": F.col("negative_mdl"),
    }

    quality_flags = F.filter(
        F.array(*[
            F.when(condition, F.lit(name))
            for name, condition in flag_rules.items()
        ]),
        lambda flag: flag.isNotNull(),
    )

    # Keep the columns that trace each row back to Bronze.
    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    return records.filter(F.col("record_eligible")).select(
        *[
            F.col(source_name).alias(output_name)
            for source_name, output_name in identifier_columns.items()
        ],
        F.concat_ws(
            "-", *[F.col(name) for name in identifier_columns]
        ).alias("monitor_id"),
        "timestamp_utc",
        F.col("ozone_value").alias("ozone_ppm"),
        F.col("Sample Measurement").alias("ozone_raw"),
        F.col("Units of Measure").alias("source_unit"),
        F.when(mdl_usable, F.col("mdl_value")).alias("mdl_ppm"),
        F.col("MDL").alias("mdl_raw"),
        "qualifier_code",
        F.col("Qualifier").alias("qualifier_raw"),
        F.col("Method Type").alias("method_type"),
        F.col("Method Code").alias("method_code"),
        F.col("Method Name").alias("method_name"),
        quality_flags.alias("quality_flags"),
        "source_occurrences",
        *lineage_columns,
    )


### 4.5 Account for excluded rows and extra copies

The audit keeps the original fields, check results, source details and outcome for each distinct record. Each representative counts once. Extra identical copies are recorded separately as `exact_duplicate`, with their number in `record_count`.

For example, three identical inputs produce one representative and an extra-copy entry counting two records. Summing `record_count` accounts for all three inputs.


In [10]:
# Build the audit using maps of original values and quality-check results.
def named_column_map(column_names):
    """Build a Spark map pairing each requested column name with its value in the row."""
    entries = []
    for name in column_names:
        entries.extend([F.lit(name), F.col(name)])
    return F.create_map(*entries)


# Build the audit with outcomes and counts for accepted records, exclusions, and extra copies.
def build_ozone_audit(records, source_columns, diagnostic_columns):
    """Build counted audit entries for representatives and duplicate copies."""
    monitor_columns = [
        "State Code", "County Code", "Site Num", "Parameter Code", "POC"
    ]

    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    base = (
        records
        .withColumn(
            "monitor_id",
            F.when(
                ~F.col("missing_identifier"),
                F.concat_ws("-", *[F.col(name) for name in monitor_columns]),
            ),
        )
        .withColumn("raw_fields", named_column_map(source_columns))
        .withColumn("field_flags", named_column_map(diagnostic_columns))
    )
    common_columns = [
        "monitor_id",
        "timestamp_utc",
        "raw_fields",
        "field_flags",
        "field_disposition",
        "field_reasons",
        "key_variant_count",
        "key_conflict",
        *lineage_columns,
    ]

    # Record each distinct row's outcome and reasons, counting it once.
    representatives = base.select(
        *common_columns,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )

    # Three identical inputs produce one representative and an extra-copy entry with count 2.
    # Summing record_count therefore still accounts for all three source records.
    duplicate_copies = (
        base
        .filter(F.col("duplicate_extra_records") > 0)
        .select(
            *common_columns,
            F.lit("exact_duplicate").alias("disposition"),
            F.concat(
                F.array(F.lit("exact_duplicate")),
                F.col("record_reasons"),
            ).alias("reasons"),
            F.col("duplicate_extra_records").alias("record_count"),
        )
    )

    return representatives.unionByName(duplicate_copies)


### 4.6 Compare ozone before and after cleaning

Calculate the same summary for the selected input and eligible observations: record counts, negative readings, unusable MDLs, ozone range and time span. Display these metrics alongside audit outcomes and retained quality flags.

The audit accounts for every selected input record, including exclusions and extra copies. Quality flags can overlap, so their counts are reported separately.


In [11]:
def summarize_ozone_values(records, ozone_column, mdl_column):
    """Return counts, value ranges and time bounds as a small Python dictionary."""
    ozone = F.col(ozone_column)
    ozone_is_finite = finite_number(ozone_column)
    mdl_is_usable = (
        finite_number(mdl_column)
        & (F.col(mdl_column) >= 0)
    )

    # Calculate one small summary row in Spark, then bring that row into Python.
    return records.agg(
        F.count("*").alias("records"),
        F.count(
            F.when(ozone_is_finite & (ozone < 0), 1)
        ).alias("negative_ozone"),
        F.count(
            F.when(~mdl_is_usable, 1)
        ).alias("missing_or_unusable_mdl"),
        F.min(
            F.when(ozone_is_finite, ozone)
        ).alias("minimum_ozone_ppm"),
        F.max(
            F.when(ozone_is_finite, ozone)
        ).alias("maximum_ozone_ppm"),
        F.date_format(
            F.min("timestamp_utc"), "yyyy-MM-dd HH:mm:ss"
        ).alias("earliest_timestamp_utc"),
        F.date_format(
            F.max("timestamp_utc"), "yyyy-MM-dd HH:mm:ss"
        ).alias("latest_timestamp_utc"),
    ).first().asDict()

def show_source_summary(summary, metrics):
    """Display the supplied before/after summaries, audit totals and retained quality flags."""
    rows = [
        (
            label,
            str(summary["before"][key])
            if summary["before"][key] is not None else "N/A",
            str(summary["after"][key])
            if summary["after"][key] is not None else "N/A",
        )
        for label, key in metrics
    ]
    
    spark.createDataFrame(
        rows,
        schema="metric STRING, before_cleaning STRING, after_cleaning STRING",
    ).show(truncate=False)
    # Audit outcomes account for the input once; retained quality flags may overlap.
    print("Exclusive source-record dispositions:")
    for disposition, count in sorted(summary["dispositions"].items()):
        print(f"  {disposition}: {count}")
    print("Reconciled source records:", sum(summary["dispositions"].values()))
    print("Overlapping quality flags on retained observations:")
    if summary["quality_flags"]:
        for flag, count in sorted(summary["quality_flags"].items()):
            print(f"  {flag}: {count}")
    else:
        print("  None")


### 4.7 Process one monitor

The caller selects the monitor's rows from Bronze and passes them to `process_ozone_source`.

The function applies the field rules, adds source details and row hashes, and handles duplicates and conflicts. It uses those results to build the observations and audit.

Before returning both tables and a summary, it checks that the audit accounts for every input record and that its accepted count matches the observation table.


In [12]:
# Process one ozone monitor and return observations, an audit, and a summary.
def process_ozone_source(raw, source, profile, environment_root):
    """Process the monitor rows selected by the caller."""
    year = profile["year"]
    monitor = profile["ozone_monitor"]
    monitor_fields = {
        "State Code": "state_code",
        "County Code": "county_code",
        "Site Num": "site_number",
        "Parameter Code": "parameter_code",
        "POC": "poc",
    }

    # Read the monitor identifiers from the profile.
    monitor_values = {
        column: monitor.get(profile_field)
        for column, profile_field in monitor_fields.items()
    }
    # Stop if the profile selects a parameter other than ozone.
    if monitor_values["Parameter Code"] != "44201":
        raise ValueError("The ozone entry point requires parameter code 44201")

    root = Path(environment_root)

    source_columns = raw.columns
    # Add a timestamp and numeric ozone and MDL values; keep the original text.
    typed = type_ozone_fields(raw)

    # Add a true/false result for each quality check to every row.
    checked = typed.select(
        "*",
        *[
            condition.alias(name)
            for name, condition in ozone_field_checks.items()
        ],
    )

    # Mark eligibility, keep all failure reasons, and use the first as the summary.
    classified = classify_ozone_fields(checked, year=year)

    # Add source details and row fingerprints to trace results back to Bronze.
    with_lineage = attach_bronze_lineage(
        classified,
        source,
        root,
        source_columns,
    )

    # Count identical copies, keep one, and mark conflicting records as ineligible.
    resolved = resolve_source_keys(
        with_lineage,
        key_columns=[*monitor_fields, "timestamp_utc"],
        key_is_usable=(
            ~F.col("missing_identifier")
            & F.col("timestamp_utc").isNotNull()
        ),
    )

    # Build the eligible observation table with quality flags and source details.
    observations = build_ozone_observations(resolved)

    # Build the audit for accepted records, exclusions, and extra copies.
    audit = build_ozone_audit(
        resolved,
        source_columns,
        list(ozone_field_checks),
    )

    # Summarize the selected input before exclusions and duplicate removal.
    before = summarize_ozone_values(
        checked,
        ozone_column="ozone_value",
        mdl_column="mdl_value",
    )

    # Summarize eligible observations after exclusions and duplicate removal.
    after = summarize_ozone_values(
        observations,
        ozone_column="ozone_ppm",
        mdl_column="mdl_ppm",
    )

    # Total source records for each audit outcome, including extra copies.
    disposition_rows = (
        audit
        .groupBy("disposition")
        .agg(F.sum("record_count").alias("source_records"))
        .orderBy("disposition")
        .collect()
    )

    # Bring only the small outcome totals into Python for the accounting checks.
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in disposition_rows
    }

    # Require all input records to be accounted for and the eligible total to match the output.
    if sum(dispositions.values()) != before["records"]:
        raise ValueError("Ozone audit counts do not reconcile with the input")

    if dispositions.get("eligible", 0) != after["records"]:
        raise ValueError("Ozone observation and eligible audit counts differ")

    # Count how many retained observations have each quality flag.
    flag_rows = (
        observations
        .select(F.explode("quality_flags").alias("quality_flag"))
        .groupBy("quality_flag")
        .agg(F.count("*").alias("observations"))
        .orderBy("quality_flag")
        .collect()
    )

    # A reading can have several flags, so these counts can overlap.
    quality_flags = {
        row["quality_flag"]: row["observations"]
        for row in flag_rows
    }

    # Return the observation table, audit table, and summary in one dictionary.
    return {
        "observations": observations,
        "audit": audit,
        "summary": {
            "monitor_id": "-".join(monitor_values.values()),
            "year": year,
            "source_version_sha256": source["download_sha256"],
            "before": before,
            "after": after,
            "dispositions": dispositions,
            "quality_flags": quality_flags,
        },
    }


### 4.8 Run the Alabama ozone check

Select McAdory's 2025 rows from Bronze and run the monitor processor. Keep the observations and audit for publication, then display the input version and cleaning summary.

We cache the selected rows because cleaning, summaries and writing reuse them. The execution plan below lets us inspect the Parquet read, monitor filter and use of cached data.


In [13]:
# Select the pinned ozone source for the profile year.
ozone_key = f"epa/hourly_ozone_{alabama_profile['year']}"
ozone_source = selected_sources[ozone_key]

# Cache the selected monitor for cleaning and summaries.
ozone_match = F.lit(True)
for column, field in {"State Code": "state_code", "County Code": "county_code",
                      "Site Num": "site_number", "Parameter Code": "parameter_code", "POC": "poc"}.items():
    ozone_match = ozone_match & (F.col(column) == alabama_profile["ozone_monitor"][field])
ozone_input = raw_tables[ozone_key].filter(ozone_match).cache()

# Build observations, an audit, and a summary for the Alabama monitor.
ozone_result = process_ozone_source(
    raw=ozone_input, source=ozone_source,
    profile=alabama_profile,
    environment_root=context.environment_root,
)
# Keep the tables for publication and the small summary for display.
ozone_observations = ozone_result["observations"]
ozone_audit = ozone_result["audit"]
ozone_result_summary = ozone_result["summary"]

# Show the selected input version and the before/after cleaning results.
print("Ozone source:", ozone_source["download_path"])
print("Source version:", ozone_result_summary["source_version_sha256"])
ozone_summary_metrics = [
    ("Records", "records"),
    ("Negative ozone values", "negative_ozone"),
    ("Missing or unusable MDL", "missing_or_unusable_mdl"),
    ("Minimum finite ozone (ppm)", "minimum_ozone_ppm"),
    ("Maximum finite ozone (ppm)", "maximum_ozone_ppm"),
    ("Earliest timestamp (UTC)", "earliest_timestamp_utc"),
    ("Latest timestamp (UTC)", "latest_timestamp_utc"),
]
show_source_summary(ozone_result_summary, ozone_summary_metrics)

ozone_input.explain(mode="formatted")

# Show a few retained readings only when extra diagnostics are enabled.
if show_diagnostics:
    ozone_observations.select(
        "monitor_id", "timestamp_utc", "ozone_ppm", "mdl_ppm", "quality_flags"
    ).orderBy("timestamp_utc").show(5, truncate=False)


Ozone source: /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/hourly_ozone_2025/2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7/hourly_44201_2025.zip
Source version: 2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7


+--------------------------+-------------------+-------------------+
|metric                    |before_cleaning    |after_cleaning     |
+--------------------------+-------------------+-------------------+
|Records                   |5166               |5071               |
|Negative ozone values     |52                 |52                 |
|Missing or unusable MDL   |0                  |0                  |
|Minimum finite ozone (ppm)|-0.001             |-0.001             |
|Maximum finite ozone (ppm)|0.072              |0.072              |
|Earliest timestamp (UTC)  |2025-02-25 01:00:00|2025-04-01 06:00:00|
|Latest timestamp (UTC)    |2025-11-03 13:00:00|2025-11-03 13:00:00|
+--------------------------+-------------------+-------------------+

Exclusive source-record dispositions:
  eligible: 5071
  qualifier_6: 95
Reconciled source records: 5166
Overlapping quality flags on retained observations:
  below_reported_mdl: 922
  negative_ozone: 52
== Physical Plan ==
AdaptiveSparkPla

### 4.9 What changed for McAdory?

Of 5,166 source records, **5,071** passed the checks. All 95 exclusions had qualifier `6`. We found no duplicate or conflicting keys in this input.

The retained values ranged from −0.001 to 0.072 ppm. All 52 negative readings were among the 922 below the reported detection limit. Adding those counts would count the negative readings twice.

The first eligible hour was 1 April at 06:00 UTC and the last was 3 November at 13:00 UTC. The hourly coverage section measures the gaps between and outside those dates.

## 5. Silver: temperature reports

### 5.1 Parse temperatures and report times

Parse NOAA temperatures as degrees Celsius and report times as UTC, keeping the original fields.

Keep the actual report time: a report at 14:53 stays at 14:53. Hourly selection happens in section 8.


In [14]:
# List the NOAA fields that describe the temperature evidence for each report.
temperature_inspection_columns = [
    "STATION",
    "DATE",
    "temperature",
    "temperature_Measurement_Code",
    "temperature_Quality_Code",
    "temperature_Report_Type",
    "temperature_Source_Code",
    "temperature_Source_Station_ID",
]


def type_temperature_fields(records):
    """Add typed UTC timestamps and temperatures, retaining source fields."""

    return (
        records
        .withColumn(
            "timestamp_utc",
            F.try_to_timestamp(
                F.col("DATE"),
                F.lit("yyyy-MM-dd'T'HH:mm:ss"),
            ),
        )
        .withColumn(
            "temperature_value",
            F.expr("try_cast(`temperature` AS DOUBLE)"),
        )
    )


# Build checks that distinguish absent text from values that failed conversion.
temperature_missing_time = missing_string("DATE")
temperature_missing_value = missing_string("temperature")

temperature_field_checks = {
    "missing_station": missing_string("STATION"),
    "missing_timestamp": temperature_missing_time,
    "invalid_timestamp": (
        ~temperature_missing_time
        & F.col("timestamp_utc").isNull()
    ),
    "missing_temperature": temperature_missing_value,
    "invalid_temperature": (
        ~temperature_missing_value
        & F.col("temperature_value").isNull()
    ),
    "nonfinite_temperature": (
        F.col("temperature_value").isNotNull()
        & ~finite_number("temperature_value")
    ),
}


### 5.2 Apply the temperature quality rules

Reports must match the selected station and year. Temperatures must be valid numbers between -100°C and 60°C, inclusive.

We accept source and quality code combinations `223/1` and `343/5`, a blank measurement code, and report types `FM12`, `FM15` or `FM16`.


In [15]:
# Classify temperature reports while retaining every source row for the audit.
def classify_temperature_fields(
    records,
    year,
    station_id,
    source_quality_pairs,
):
    """Add eligibility and failure reasons using the selected station, year and NOAA policy."""
    # Read the profile policy and restrict it to source/quality pairs we have reviewed.
    accepted_pairs = {
        (str(source).strip(), str(quality).strip())
        for source, quality in source_quality_pairs
    }
    reviewed_pairs = {("223", "1"), ("343", "5")}

    if not accepted_pairs or not accepted_pairs <= reviewed_pairs:
        raise ValueError(
            "Temperature source/quality pairs must be a nonempty subset "
            "of the reviewed pairs: 223/1 and 343/5."
        )

    # Normalize metadata for comparison while retaining the source fields.
    source = F.coalesce(
        F.trim(F.col("temperature_Source_Code")), F.lit("")
    )
    quality = F.coalesce(
        F.trim(F.col("temperature_Quality_Code")), F.lit("")
    )
    report = F.coalesce(
        F.trim(F.col("temperature_Report_Type")), F.lit("")
    )
    measurement = F.coalesce(
        F.trim(F.col("temperature_Measurement_Code")), F.lit("")
    )

    timestamp = F.col("timestamp_utc")
    value = F.col("temperature_value")
    finite = finite_number("temperature_value")

    # Match the source and quality code together, because quality codes depend on the source.
    matches_pair = F.lit(False)
    for pair_source, pair_quality in sorted(accepted_pairs):
        matches_pair = matches_pair | (
            (source == pair_source) & (quality == pair_quality)
        )

    # The same quality code can mean different things for different NOAA sources.
    rejected_quality = (
        ((source == "343") & quality.isin("2", "3", "6", "7"))
        | ((source == "223") & quality.isin("2", "3", "5"))
    )

    # Check station, time, measurement and metadata. The first failure is the summary reason.
    rules = [
        ("missing_station", F.col("missing_station")),
        (
            "unexpected_station",
            ~F.col("missing_station")
            & (F.col("STATION") != F.lit(station_id)),
        ),
        ("missing_timestamp", F.col("missing_timestamp")),
        ("invalid_timestamp", F.col("invalid_timestamp")),
        (
            "outside_year",
            timestamp.isNotNull() & (F.year(timestamp) != year),
        ),
        ("missing_temperature", F.col("missing_temperature")),
        ("invalid_temperature", F.col("invalid_temperature")),
        ("nonfinite_temperature", F.col("nonfinite_temperature")),
        (
            "outside_temperature_range",
            finite & ~value.between(-100.0, 60.0),
        ),
        ("rejected_source_quality", rejected_quality),
        (
            "unsupported_source_quality",
            ~matches_pair & ~rejected_quality,
        ),
        (
            "unsupported_report_type",
            ~report.isin("FM12", "FM15", "FM16"),
        ),
        ("unsupported_measurement_code", measurement != ""),
    ]

    # Keep all applicable failure reasons, leaving out checks that passed.
    reasons = F.filter(
        F.array(*[
            F.when(condition, F.lit(reason))
            for reason, condition in rules
        ]),
        lambda reason: reason.isNotNull(),
    )

    # Return normalised codes, all reasons and the eligibility decision without dropping rows.
    return (
        records
        .withColumn("source_code", source)
        .withColumn("quality_code", quality)
        .withColumn("report_type", report)
        .withColumn("measurement_code", measurement)
        .withColumn("field_reasons", reasons)
        .withColumn("field_eligible", F.size("field_reasons") == 0)
        .withColumn(
            "field_disposition",
            F.when(F.col("field_eligible"), F.lit("eligible"))
            .otherwise(F.element_at("field_reasons", 1)),
        )
    )


### 5.3 Resolve reports at the same timestamp

Reports at the same station and time can be exact copies, agree on temperature while differing in other weather fields, or contain conflicting temperature information. Agreement includes the temperature value and its associated NOAA codes.

We count exact copies together. When the temperature fields agree, we keep one report and account for the others in the audit. When they conflict, the reports are excluded. A fixed ordering makes repeated runs choose the same report.


In [16]:
# Resolve repeated station/timestamp keys using the temperature evidence.
def resolve_temperature_keys(records, source_columns, temperature_columns):
    """Count copies, reject conflicting temperature evidence and flag redundant reports."""
    # Only compare reports from the same source file and version.
    source_scope = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
    ]
    key_columns = ["STATION", "timestamp_utc"]

    # A missing station or timestamp cannot form a usable report key.
    usable_key = F.coalesce(
        ~F.col("missing_station")
        & F.col("timestamp_utc").isNotNull(),
        F.lit(False),
    )

    # Collapse complete copies, including their stable derived fields.
    distinct_records = records.groupBy(*records.columns).agg(
        F.count("*").alias("source_occurrences")
    )

    key_window = Window.partitionBy(*(source_scope + key_columns))
    # Compare the temperature value AND its source, quality and report metadata.
    # Equal temperatures alone do not mean two reports provide the same evidence.
    temperature_window = key_window.orderBy(
        *[F.col(name) for name in temperature_columns]
    )

    # Choose the same source row each time, using its hash and full contents.
    source_json = F.to_json(
        F.struct(*[F.col(name) for name in sorted(source_columns)]),
        options={"ignoreNullFields": "false"},
    )
    representative_window = key_window.orderBy(
        F.col("source_record_sha256"),
        source_json,
    )

    # dense_rank gives equal temperature evidence the same rank.
    # row_number chooses one full source row when the evidence agrees.
    ranked = (
        distinct_records
        .withColumn(
            "_temperature_rank",
            F.dense_rank().over(temperature_window),
        )
        .withColumn(
            "_source_rank",
            F.row_number().over(representative_window),
        )
    )

    # The highest dense rank tells us how many different temperature variants exist.
    # Exclude conflicting variants. When eligible reports agree, keep the first
    # representative and label the others redundant_temperature for the audit.
    resolved = (
        ranked
        .withColumn(
            "duplicate_extra_records",
            F.col("source_occurrences") - 1,
        )
        .withColumn(
            "key_variant_count",
            F.when(
                usable_key,
                F.max("_temperature_rank").over(key_window),
            ),
        )
        .withColumn(
            "key_source_records",
            F.when(
                usable_key,
                F.sum("source_occurrences").over(key_window),
            ),
        )
        .withColumn(
            "key_conflict",
            F.coalesce(
                F.col("key_variant_count") > 1,
                F.lit(False),
            ),
        )
        .withColumn(
            "redundant_temperature",
            usable_key
            & ~F.col("key_conflict")
            & F.col("field_eligible")
            & (F.col("_source_rank") > 1),
        )
        .withColumn(
            "record_disposition",
            F.when(
                F.col("key_conflict"),
                F.lit("conflicting_key"),
            )
            .when(
                F.col("redundant_temperature"),
                F.lit("redundant_temperature"),
            )
            .otherwise(F.col("field_disposition")),
        )
        .withColumn(
            "record_reasons",
            F.when(
                F.col("key_conflict"),
                F.concat(
                    F.array(F.lit("conflicting_key")),
                    F.col("field_reasons"),
                ),
            )
            .when(
                F.col("redundant_temperature"),
                F.concat(
                    F.array(F.lit("redundant_temperature")),
                    F.col("field_reasons"),
                ),
            )
            .otherwise(F.col("field_reasons")),
        )
        .withColumn(
            "record_eligible",
            F.col("record_disposition") == "eligible",
        )
        .drop("_temperature_rank", "_source_rank")
    )

    # Return the decisions and copy counts for both the observation table and audit.
    return resolved


### 5.4 Build the temperature and audit tables

Keep eligible reports with their temperature, original timestamp, quality flags and Bronze lineage. The audit accounts for accepted and rejected records, redundant reports and extra identical copies.


In [17]:
# Build the eligible report table and the separate audit from the same resolved rows.
def build_temperature_observations(records):
    """Return eligible temperature reports with their original times, flags and source details."""
    # A missing temperature source-station ID is flagged without excluding the report.
    missing_original_station = missing_string(
        "temperature_Source_Station_ID"
    )

    quality_flags = F.filter(
        F.array(
            F.when(
                missing_original_station,
                F.lit("missing_source_station_id"),
            ),
        ),
        lambda flag: flag.isNotNull(),
    )

    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    # Keep the actual report time, Celsius value, original fields and Bronze source details.
    return records.filter(F.col("record_eligible")).select(
        F.col("STATION").alias("station_id"),
        "timestamp_utc",
        F.col("temperature_value").alias("temperature_c"),
        F.col("temperature").alias("temperature_raw"),
        F.col("DATE").alias("timestamp_raw"),
        "source_code",
        "quality_code",
        "report_type",
        "measurement_code",
        F.when(
            ~missing_original_station,
            F.trim(F.col("temperature_Source_Station_ID")),
        ).alias("source_station_id"),
        F.col("temperature_Source_Code").alias("source_code_raw"),
        F.col("temperature_Quality_Code").alias("quality_code_raw"),
        F.col("temperature_Report_Type").alias("report_type_raw"),
        F.col("temperature_Measurement_Code").alias("measurement_code_raw"),
        F.col("temperature_Source_Station_ID").alias("source_station_id_raw"),
        quality_flags.alias("quality_flags"),
        "source_occurrences",
        F.col("key_source_records").alias("supporting_source_records"),
        *lineage_columns,
    )


def build_temperature_audit(records, source_columns, diagnostic_columns):
    """Build counted audit entries for source rows and duplicate copies."""
    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    # Attach raw field values and check results so each audit entry can be explained.
    base = (
        records
        .withColumn(
            "station_id",
            F.when(
                ~F.col("missing_station"),
                F.col("STATION"),
            ),
        )
        .withColumn(
            "raw_fields",
            named_column_map(source_columns),
        )
        .withColumn(
            "field_flags",
            named_column_map(diagnostic_columns),
        )
    )

    common_columns = [
        "station_id",
        "timestamp_utc",
        "raw_fields",
        "field_flags",
        "field_disposition",
        "field_reasons",
        "key_variant_count",
        "key_source_records",
        "key_conflict",
        "redundant_temperature",
        *lineage_columns,
    ]

    # Count each distinct source row once, whether accepted or excluded.
    representatives = base.select(
        *common_columns,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )

    # Account for extra identical copies separately using their copy counts.
    duplicate_copies = (
        base
        .filter(F.col("duplicate_extra_records") > 0)
        .select(
            *common_columns,
            F.lit("exact_duplicate").alias("disposition"),
            F.concat(
                F.array(F.lit("exact_duplicate")),
                F.col("record_reasons"),
            ).alias("reasons"),
            F.col("duplicate_extra_records").alias("record_count"),
        )
    )

    return representatives.unionByName(duplicate_copies)


### 5.5 Process one station

`process_temperature_source` applies the station, year and quality rules to rows from Bronze. It returns observations, an audit and before/after summaries of counts, temperatures and time bounds.

The audit must account for every input record and agree with the eligible observation count.


In [18]:
# Define the summary and the main station-processing function used below.
def summarize_temperature_values(
    records,
    temperature_column,
    source_station_column,
):
    """Summarize temperature values, metadata completeness and time bounds."""
    temperature = F.col(temperature_column)
    temperature_is_finite = finite_number(temperature_column)

    # Calculate comparable before/after counts, ranges and the observed time span.
    return records.agg(
        F.count("*").alias("records"),
        F.count(
            F.when(~temperature_is_finite, 1)
        ).alias("missing_or_unusable_temperature"),
        F.count(
            F.when(
                temperature_is_finite & (temperature < 0),
                1,
            )
        ).alias("negative_temperatures"),
        F.count(
            F.when(missing_string(source_station_column), 1)
        ).alias("missing_source_station_id"),
        F.min(
            F.when(temperature_is_finite, temperature)
        ).alias("minimum_temperature_c"),
        F.max(
            F.when(temperature_is_finite, temperature)
        ).alias("maximum_temperature_c"),
        F.date_format(
            F.min("timestamp_utc"),
            "yyyy-MM-dd HH:mm:ss",
        ).alias("earliest_timestamp_utc"),
        F.date_format(
            F.max("timestamp_utc"),
            "yyyy-MM-dd HH:mm:ss",
        ).alias("latest_timestamp_utc"),
    ).first().asDict()


def process_temperature_source(raw, source, profile, environment_root):
    """Build temperature observations, audit records and quality summaries."""
    # Read the selected year, station and temperature policy from the profile.
    year = profile["year"]
    station_id = profile["weather_mapping"]["noaa_station_id"]
    source_quality_pairs = (
        profile["temperature_policy"]["source_quality_pairs"]
    )

    root = Path(environment_root)

    # Fields compared when resolving reports at the same timestamp.
    temperature_columns = [
        "STATION",
        "DATE",
        "temperature",
        "temperature_Measurement_Code",
        "temperature_Quality_Code",
        "temperature_Report_Type",
        "temperature_Source_Code",
        "temperature_Source_Station_ID",
    ]

    source_columns = raw.columns

    # Keep every source row while adding typed values and check results; build the audit later.
    typed = type_temperature_fields(raw)
    checked = typed.select(
        "*",
        *[
            condition.alias(name)
            for name, condition in temperature_field_checks.items()
        ],
    )

    # Apply the station, year, value and NOAA metadata rules.
    classified = classify_temperature_fields(
        checked,
        year=year,
        station_id=station_id,
        source_quality_pairs=source_quality_pairs,
    )

    # Attach the source-file identity and original-row fingerprint.
    with_lineage = attach_bronze_lineage(
        classified,
        source=source,
        environment_root=root,
        source_columns=source_columns,
    )

    # Resolve exact copies, conflicting evidence and redundant reports at the same time.
    resolved = resolve_temperature_keys(
        with_lineage,
        source_columns=source_columns,
        temperature_columns=temperature_columns,
    )

    # Build both outputs from the same decisions so accepted and excluded records reconcile.
    observations = build_temperature_observations(resolved)
    audit = build_temperature_audit(
        resolved,
        source_columns=source_columns,
        diagnostic_columns=list(temperature_field_checks),
    )

    # Compare the full input with the retained observations using the same metrics.
    before = summarize_temperature_values(
        checked,
        temperature_column="temperature_value",
        source_station_column="temperature_Source_Station_ID",
    )
    after = summarize_temperature_values(
        observations,
        temperature_column="temperature_c",
        source_station_column="source_station_id",
    )

    # Sum record_count so an audit entry representing several copies contributes all of them.
    disposition_rows = (
        audit
        .groupBy("disposition")
        .agg(F.sum("record_count").alias("source_records"))
        .orderBy("disposition")
        .collect()
    )
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in disposition_rows
    }

    # Require complete audit accounting and the eligible output count.
    if sum(dispositions.values()) != before["records"]:
        raise ValueError(
            "Temperature audit counts do not reconcile with the input."
        )

    if dispositions.get("eligible", 0) != after["records"]:
        raise ValueError(
            "Temperature observation and eligible audit counts differ."
        )

    if after["missing_or_unusable_temperature"] != 0:
        raise ValueError(
            "Eligible observations contain unusable temperatures."
        )

    # Count retained quality flags and compare the missing-source-station count with the summary.
    flag_rows = (
        observations
        .select(F.explode("quality_flags").alias("quality_flag"))
        .groupBy("quality_flag")
        .agg(F.count("*").alias("observations"))
        .orderBy("quality_flag")
        .collect()
    )
    quality_flags = {
        row["quality_flag"]: row["observations"]
        for row in flag_rows
    }

    if (
        quality_flags.get("missing_source_station_id", 0)
        != after["missing_source_station_id"]
    ):
        raise ValueError(
            "Missing source-station flags and summary counts differ."
        )

    # Return the two Spark tables and a small summary for display and validation.
    return {
        "observations": observations,
        "audit": audit,
        "summary": {
            "station_id": station_id,
            "year": year,
            "source_version_sha256": source["download_sha256"],
            "before": before,
            "after": after,
            "dispositions": dispositions,
            "quality_flags": quality_flags,
        },
    }


### 5.6 Run the Alabama temperature check

Select Alabaster's 2025 reports from Bronze, run the station processor and display the cleaning summary.


In [19]:
temperature_station_id = alabama_profile["weather_mapping"]["noaa_station_id"]
temperature_key = f"noaa/hourly_weather_{alabama_profile['year']}_{temperature_station_id}"
temperature_source = selected_sources[temperature_key]

temperature_input = raw_tables[temperature_key].cache()

temperature_result = process_temperature_source(
    raw=temperature_input,
    source=temperature_source,
    profile=alabama_profile,
    environment_root=context.environment_root,
)
temperature_observations = temperature_result["observations"]
temperature_audit = temperature_result["audit"]
temperature_result_summary = temperature_result["summary"]

print("Temperature source:", temperature_source["download_path"])
print("Source version:", temperature_result_summary["source_version_sha256"])
temperature_summary_metrics = [
    ("Records", "records"),
    ("Missing or unusable temperature", "missing_or_unusable_temperature"),
    ("Temperatures below zero (C)", "negative_temperatures"),
    ("Missing original source-station ID", "missing_source_station_id"),
    ("Minimum finite temperature (C)", "minimum_temperature_c"),
    ("Maximum finite temperature (C)", "maximum_temperature_c"),
    ("Earliest timestamp (UTC)", "earliest_timestamp_utc"),
    ("Latest timestamp (UTC)", "latest_timestamp_utc"),
]
show_source_summary(temperature_result_summary, temperature_summary_metrics)

# Optionally display sample reports; their timestamps have not been rounded to hours.
if show_diagnostics:
    temperature_observations.select(
        "station_id", "timestamp_utc", "temperature_c",
        "source_code", "quality_code", "report_type", "quality_flags",
    ).orderBy("timestamp_utc").show(5, truncate=False)


26/11/20 16:13:28 WARN DAGScheduler: Broadcasting large task binary with size 1012.7 KiB
26/11/20 16:13:28 WARN DAGScheduler: Broadcasting large task binary with size 1013.4 KiB
26/11/20 16:13:28 WARN DAGScheduler: Broadcasting large task binary with size 1009.4 KiB


Temperature source: /home/ubuntu/dat535-final-project-data/dev/data/bronze/noaa/hourly_weather_2025_USW00053864/2ccd886d5e6ba9b8c502058db047094dc8a76e50bcfd1c3d54f5c458ab97d308/GHCNh_USW00053864_2025.psv
Source version: 2ccd886d5e6ba9b8c502058db047094dc8a76e50bcfd1c3d54f5c458ab97d308
+----------------------------------+-------------------+-------------------+
|metric                            |before_cleaning    |after_cleaning     |
+----------------------------------+-------------------+-------------------+
|Records                           |11851              |11686              |
|Missing or unusable temperature   |29                 |0                  |
|Temperatures below zero (C)       |365                |365                |
|Missing original source-station ID|29                 |0                  |
|Minimum finite temperature (C)    |-10.0              |-10.0              |
|Maximum finite temperature (C)    |36.7               |36.7               |
|Earliest timestamp (U

### 5.7 What changed for Alabaster?

Of 11,851 weather reports, **11,686** passed. There were 29 missing temperatures, 7 rejected source/quality combinations and 129 unsupported combinations. We found no repeated combinations of station and timestamp in this file.

The retained temperatures ranged from −10.0°C to 36.7°C, including 365 readings below zero. Several reports can fall in the same hour, so 11,686 reports represent fewer than 11,686 hours.

## 6. Silver: site, monitor and station records

### 6.1 Select the three reference records

Use the profile IDs to find the selected EPA site and monitor and the NOAA weather station.

These records describe the locations and instruments. The following steps check them and keep an audit of the selected rows.


In [20]:
# Use the Alabama profile to select its site, ozone monitor and weather station.
reference_site_keys = {
    "State Code": alabama_profile["ozone_monitor"]["state_code"],
    "County Code": alabama_profile["ozone_monitor"]["county_code"],
    "Site Number": alabama_profile["ozone_monitor"]["site_number"],
}

# Define each reference source and the identifiers used to select its records.
# The monitor key adds parameter and POC; the weather station uses its GHCN ID.
reference_specs = [
    (
        "epa_sites",
        "epa",
        "sites",
        reference_site_keys,
    ),
    (
        "epa_monitors",
        "epa",
        "monitors",
        {
            **reference_site_keys,
            "Parameter Code": (
                alabama_profile["ozone_monitor"]["parameter_code"]
            ),
            "POC": alabama_profile["ozone_monitor"]["poc"],
        },
    ),
    (
        "noaa_stations",
        "noaa",
        "stations",
        {
            "GHCN_ID": (
                alabama_profile["weather_mapping"]["noaa_station_id"]
            ),
        },
    ),
]

# Keep selected-source metadata and the small profile-selected reference inputs.
reference_sources = {}
reference_profile_rows = {}

# Select the profile rows from each Bronze reference table.
for (
    reference_name,
    reference_provider,
    reference_dataset,
    reference_keys,
) in reference_specs:
    reference_key = f"{reference_provider}/{reference_dataset}"
    reference_source = selected_sources[reference_key]
    reference_records = raw_tables[reference_key]

    # Build a filter requiring every identifier to match the selected profile.
    reference_match = F.lit(True)
    for source_column, expected_value in reference_keys.items():
        reference_match = reference_match & (
            F.col(source_column) == F.lit(expected_value)
        )

    # Keep the Bronze source details for tracing the cleaned records back to their input.
    reference_sources[reference_name] = reference_source

    # Keep all matching rows for cleaning; duplicate and conflicting records are handled later.
    reference_profile_rows[reference_name] = reference_records.filter(reference_match).cache()


### 6.2 Parse coordinates, elevations and dates

Convert reference fields to numbers and dates while keeping the original values. Build checks for missing values, failed conversions, coordinate ranges and inconsistent dates. The next step uses these checks to decide eligibility and quality flags.


In [21]:
# Define each new column's name, original source column, and required data type.
reference_type_specs = {
    "epa_sites": {
        "latitude_deg": ("Latitude", "double"),
        "longitude_deg": ("Longitude", "double"),
        "elevation_m": ("Elevation", "double"),
        "site_established_date": ("Site Established Date", "date"),
        "site_closed_date": ("Site Closed Date", "date"),
        "extraction_date": ("Extraction Date", "date"),
    },
    "epa_monitors": {
        "latitude_deg": ("Latitude", "double"),
        "longitude_deg": ("Longitude", "double"),
        "first_year_of_data": ("First Year of Data", "int"),
        "last_sample_date": ("Last Sample Date", "date"),
        "extraction_date": ("Extraction Date", "date"),
    },
    "noaa_stations": {
        "latitude_deg": ("LATITUDE", "double"),
        "longitude_deg": ("LONGITUDE", "double"),
        "elevation_m": ("ELEVATION", "double"),
    },
}

# Add numeric and date columns while keeping the original source fields.
def type_reference_fields(records, reference_name):
    """Add typed reference fields while preserving original source columns."""

    # Use the conversion rules for this reference type.
    field_specs = reference_type_specs[reference_name]

    typed_expressions = []

    for output_name, (source_name, target_type) in field_specs.items():
        if target_type == "date":
            # Parse dates in year-month-day format; invalid values become null.
            parsed_value = F.try_to_timestamp(
                F.col(source_name),
                F.lit("yyyy-MM-dd"),
            ).cast("date")
        elif target_type in {"double", "int"}:
            # Escape backticks in the column name for the SQL expression.
            # Text that cannot be converted becomes null.
            escaped_source_name = source_name.replace("`", "``")
            parsed_value = F.expr(
                f"try_cast(`{escaped_source_name}` AS {target_type})"
            )
        # Stop if the conversion rules request an unsupported data type.
        else:
            raise ValueError(
                f"Unsupported reference field type: {target_type}"
            )

        typed_expressions.append(
            parsed_value.alias(output_name)
        )

    return records.select("*", *typed_expressions)


# Define named quality-check conditions for a reference source without reading its rows.
def build_reference_field_checks(reference_name, key_columns):
    """Build reference diagnostics without scanning source records."""
    # Create a missing-value check for each part of the reference identifier.
    checks = {
        f"missing_identifier_{name.lower().replace(' ', '_')}": missing_string(name)
        for name in key_columns
    }

    # Compare source values with typed values to distinguish missing data from failed conversions.
    for typed_column, (source_column, data_type) in (
        reference_type_specs[reference_name].items()
    ):
        source_missing = missing_string(source_column)

        checks[f"missing_{typed_column}"] = source_missing
        checks[f"invalid_{typed_column}"] = (
            ~source_missing & F.col(typed_column).isNull()
        )

        # Flag numeric values that are NaN or infinity.
        if data_type == "double":
            checks[f"nonfinite_{typed_column}"] = (
                F.col(typed_column).isNotNull()
                & ~finite_number(typed_column)
            )

    # Check that finite coordinates fall within valid latitude and longitude ranges.
    checks["latitude_out_of_range"] = (
        finite_number("latitude_deg")
        & ~F.col("latitude_deg").between(-90.0, 90.0)
    )
    checks["longitude_out_of_range"] = (
        finite_number("longitude_deg")
        & ~F.col("longitude_deg").between(-180.0, 180.0)
    )

    # Check whether EPA records are missing the coordinate reference system.
    if reference_name in {"epa_sites", "epa_monitors"}:
        checks["missing_datum"] = missing_string("Datum")

    # Check that site dates are consistent with each other and the extraction date.
    # Reference dates may fall outside the year being analysed.
    if reference_name == "epa_sites":
        checks["closure_before_establishment"] = (
            F.col("site_closed_date") < F.col("site_established_date")
        )
        checks["establishment_after_extraction"] = (
            F.col("site_established_date") > F.col("extraction_date")
        )
        checks["closure_after_extraction"] = (
            F.col("site_closed_date") > F.col("extraction_date")
        )

    # Check that the first data year, last sample date, and extraction date are in order.
    elif reference_name == "epa_monitors":
        checks["first_year_after_last_sample"] = (
            F.col("first_year_of_data") > F.year("last_sample_date")
        )
        checks["last_sample_after_extraction"] = (
            F.col("last_sample_date") > F.col("extraction_date")
        )
    return checks


reference_field_checks = {
    reference_name: build_reference_field_checks(reference_name, reference_keys)
    for reference_name, _, _, reference_keys in reference_specs
}


### 6.3 Decide which reference records are usable

Missing identifiers or unusable coordinates exclude a record. Problems in optional fields are recorded as flags.

NOAA uses `-999.9` for missing elevation. EPA coordinates not marked WGS84 need review before we use them in the distance calculation.


In [22]:
# Classify reference records using required-field checks and optional metadata flags.
def classify_reference_fields(records, reference_name):
    """Add exclusion reasons and optional quality flags to reference records."""
    # Copy the shared checks so source-specific additions do not change the original dictionary.
    checks = dict(reference_field_checks[reference_name])

    # Treat blank elevation and NOAA's -999.9 missing-value marker as missing.
    if reference_name == "noaa_stations":
        checks["missing_elevation_m"] = (
            missing_string("ELEVATION")
            | F.coalesce(
                F.col("elevation_m") == F.lit(-999.9),
                F.lit(False),
            )
        )

    # Keep the stated coordinate reference system (datum). Flag non-WGS84 coordinates
    # for review before using them to judge the distance between the stations.
    if reference_name in {"epa_sites", "epa_monitors"}:
        checks["non_wgs84_datum"] = (
            ~missing_string("Datum")
            & (F.upper(F.trim(F.col("Datum"))) != F.lit("WGS84"))
        )

    # Flag first data years outside 1–9999 or later than the reference extraction year.
    if reference_name == "epa_monitors":
        checks["first_year_out_of_range"] = (
            F.col("first_year_of_data").isNotNull()
            & ~F.col("first_year_of_data").between(1, 9999)
        )
        checks["first_year_after_extraction"] = (
            F.col("first_year_of_data") > F.year("extraction_date")
        )

    # Missing identifier parts and unusable coordinates make a reference record ineligible.
    exclusion_names = [
        name
        for name in checks
        if name.startswith("missing_identifier_")
    ] + [
        "missing_latitude_deg",
        "invalid_latitude_deg",
        "nonfinite_latitude_deg",
        "latitude_out_of_range",
        "missing_longitude_deg",
        "invalid_longitude_deg",
        "nonfinite_longitude_deg",
        "longitude_out_of_range",
    ]

    # Keep other findings as quality flags without excluding the record.
    # An absent site closure date is allowed without a quality flag.
    optional_flag_names = [
        name
        for name in checks
        if name not in exclusion_names
        and name != "missing_site_closed_date"
    ]

    # Collect the names of checks that identify a problem in each row.
    def matching_checks(names):
        """Return the names of the selected checks that apply to each row."""
        return F.filter(
            # Include a check's name when its condition is true; treat null conditions as false.
            F.array(
                *[
                    F.when(
                        F.coalesce(checks[name], F.lit(False)),
                        F.lit(name),
                    )
                    for name in names
                ]
            ),
            # Remove empty entries so the array contains only detected problems.
            lambda reason: reason.isNotNull(),
        )

    # Record every check as true or false, treating null as false.
    diagnostic_entries = []
    for name, condition in checks.items():
        diagnostic_entries.extend(
            [
                F.lit(name),
                F.coalesce(condition, F.lit(False)),
            ]
        )

    # Keep every source row and add the classification and diagnostic columns.
    return (
        records
        .withColumn(
            "reference_field_flags",
            F.create_map(*diagnostic_entries),
        )
        .withColumn(
            "field_reasons",
            matching_checks(exclusion_names),
        )
        .withColumn(
            "field_eligible",
            F.size("field_reasons") == 0,
        )
        # Use "eligible" for usable records; otherwise use the first exclusion reason as the summary.
        .withColumn(
            "field_disposition",
            F.when(
                F.col("field_eligible"),
                F.lit("eligible"),
            ).otherwise(F.element_at("field_reasons", 1)),
        )
        .withColumn(
            "reference_quality_flags",
            matching_checks(optional_flag_names),
        )
    )

### 6.4 Resolve duplicate identifiers

Count exact copies together. Different records sharing the same site, monitor or station identifier within a source file and version are excluded from the eligible output and kept in the audit.

The helper temporarily uses sorted map entries when grouping exact copies.


In [23]:
# Reuse the source duplicate rules for site, monitor and station identifiers.
# Count identical reference records and identify conflicts at the same reference key.
def resolve_reference_keys(records, key_columns):
    """Count identical copies and flag conflicting rows for the same reference identifier."""
    key_columns = list(key_columns)

    # Treat a key as usable only when every identifier part is present.
    key_is_usable = F.lit(True)
    for column in key_columns:
        key_is_usable = key_is_usable & ~missing_string(column)

    # Spark cannot group by map columns. Convert each map to sorted key/value entries
    # so maps with the same contents compare equally regardless of entry order.
    groupable = (
        records
        .withColumn(
            "__reference_field_entries",
            F.sort_array(F.map_entries("reference_field_flags")),
        )
        .drop("reference_field_flags")
    )

    # Count identical copies and mark conflicting records as ineligible.
    # Compare keys only within the same source file and version.
    return (
        resolve_source_keys(
            groupable,
            key_columns=key_columns,
            key_is_usable=key_is_usable,
        )
        # Restore the quality-check results to their original map format for the audit.
        .withColumn(
            "reference_field_flags",
            F.map_from_entries("__reference_field_entries"),
        )
        # Remove the temporary column used to compare the maps.
        .drop("__reference_field_entries")
    )

### 6.5 Build the reference tables and audit

Keep usable reference records in the three output tables. Their shared audit keeps the original values, quality flags and outcomes, and accounts for duplicate copies.


In [24]:
# Define identifiers and readable output names for the three reference tables.
reference_output_specs = {
    "epa_sites": {
        "id_column": "site_id",
        "keys": {
            "State Code": "state_code", "County Code": "county_code",
            "Site Number": "site_number",
        },
        "text": {
            "Local Site Name": "site_name", "State Name": "state_name",
            "County Name": "county_name", "Land Use": "land_use",
            "Location Setting": "location_setting", "Datum": "datum",
        },
    },
    "epa_monitors": {
        "id_column": "monitor_id",
        "keys": {
            "State Code": "state_code", "County Code": "county_code",
            "Site Number": "site_number", "Parameter Code": "parameter_code",
            "POC": "poc",
        },
        "text": {
            "Parameter Name": "parameter_name", "Monitor Type": "monitor_type",
            "Measurement Scale": "measurement_scale", "Datum": "datum",
            "Last Method Code": "last_method_code", "Last Method": "last_method",
        },
    },
    "noaa_stations": {
        "id_column": "station_id",
        "keys": {"GHCN_ID": "station_id"},
        "text": {
            "NAME": "station_name", "STATE": "state_abbreviation",
            "ICAO": "icao_code", "ISO_CODE": "country_code",
        },
    },
}

# Keep source details and row hashes in both outputs.
reference_lineage_columns = [
    "source_provider", "source_dataset", "source_version_sha256",
    "source_file_path", "source_file_sha256", "source_manifest_path",
    "source_record_hash_scheme", "source_record_sha256",
]


# Build the eligible reference table and an audit accounting for every selected source record.
def build_reference_outputs(records, reference_name, source_columns):
    """Build an eligible reference table and a counted source audit."""
    spec = reference_output_specs[reference_name]

    # Require every part of the identifier to be present and nonblank.
    usable_key = F.lit(True)
    for column in spec["keys"]:
        usable_key = usable_key & ~missing_string(column)

    # Join the identifier parts with hyphens; leave the ID null if any part is missing.
    entity_id = F.when(
        usable_key,
        F.concat_ws("-", *[F.col(column) for column in spec["keys"]]),
    )

    identifiers = [
        F.col(original).alias(output)
        for original, output in spec["keys"].items()
        if output != spec["id_column"]
    ]
    identifiers.append(entity_id.alias(spec["id_column"]))

    # Keep required coordinates; replace unusable optional typed values with null.
    typed_fields = []
    for column in reference_type_specs[reference_name]:
        value = F.col(column)
        if column not in {"latitude_deg", "longitude_deg"}:
            unusable = F.lit(False)
            # Combine the flags for missing values, failed conversions, NaN and infinity.
            # Treat a missing flag as false.
            for prefix in ("missing_", "invalid_", "nonfinite_"):
                unusable = unusable | F.coalesce(
                    F.col("reference_field_flags")[prefix + column],
                    F.lit(False),
                )
            # Also mark the first data year as unusable if it is outside the allowed range.
            if column == "first_year_of_data":
                unusable = unusable | F.coalesce(
                    F.col("reference_field_flags")["first_year_out_of_range"],
                    F.lit(False),
                )
            # Use null for an unusable optional value; otherwise keep the converted value.
            value = F.when(~unusable, value)
        typed_fields.append(value.alias(column))

    # Trim optional text and use null when a field is blank or absent.
    text_fields = []
    for original, output in spec["text"].items():
        value = (
            F.when(~missing_string(original), F.trim(F.col(original)))
            if original in records.columns
            else F.lit(None).cast("string")
        )
        text_fields.append(value.alias(output))

    # Build the eligible reference table with flags and source details.
    reference = records.filter("record_eligible").select(
        *identifiers,
        *typed_fields,
        *text_fields,
        F.col("reference_quality_flags").alias("quality_flags"),
        "source_occurrences",
        *reference_lineage_columns,
    )

    # Keep raw fields and check results for eligible and excluded reference records.
    audit_base = (
        records
        .withColumn("reference", F.lit(reference_name))
        .withColumn("entity_id", entity_id)
        .withColumn("raw_fields", named_column_map(source_columns))
        .withColumn("field_flags", F.col("reference_field_flags"))
    )

    # Keep the same evidence columns for distinct records and their extra copies.
    common = [
        "reference", "entity_id", "raw_fields", "field_flags",
        "field_disposition", "field_reasons", "reference_quality_flags",
        "key_variant_count", "key_conflict", *reference_lineage_columns,
    ]
    # Count each distinct source record once in the audit.
    representatives = audit_base.select(
        *common,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )
    # Add the extra copies separately so every selected input record is accounted for.
    duplicate_copies = audit_base.filter("duplicate_extra_records > 0").select(
        *common,
        F.lit("exact_duplicate").alias("disposition"),
        F.concat(
            F.array(F.lit("exact_duplicate")), F.col("record_reasons"),
        ).alias("reasons"),
        F.col("duplicate_extra_records").alias("record_count"),
    )

    return {
        "reference": reference,
        "audit": representatives.unionByName(duplicate_copies),
    }

### 6.6 Check the reference totals

The selected input count must match the number of distinct records plus extra copies. Every distinct record must then be accounted for as accepted, conflicting or excluded by the field checks.

The accepted count in the audit must match the reference output table.


In [25]:
# Apply one shared processing flow to the selected rows of a reference type.
def process_reference_records(records, source, reference_name, environment_root):
    """Process selected raw reference records and reconcile their audit."""
    # Remember the selected input count, then type, classify and trace the records to Bronze.
    input_records = records.count()
    source_columns = records.columns

    # Convert the reference values to numbers and dates, keeping the original fields.
    typed = type_reference_fields(records, reference_name)

    # Mark records that fail required checks and flag problems with optional fields.
    classified = classify_reference_fields(typed, reference_name)

    # Attach the Bronze file details and original-row fingerprint for tracing the results.
    with_lineage = attach_bronze_lineage(
        classified, source, environment_root, source_columns,
    )
    # Resolve duplicates once; the counts and both outputs use this same result.
    resolved = resolve_reference_keys(
        with_lineage,
        key_columns=list(reference_output_specs[reference_name]["keys"]),
    )
    # Count original occurrences, distinct rows, extra copies and the three distinct-row outcomes.
    counts = resolved.agg(
        F.coalesce(F.sum("source_occurrences"), F.lit(0)).alias(
            "input_records"
        ),
        F.count("*").alias("distinct_source_rows"),
        F.coalesce(F.sum("duplicate_extra_records"), F.lit(0)).alias(
            "duplicate_extra_records"
        ),
        F.count(F.when(F.col("record_eligible"), 1)).alias(
            "eligible_records"
        ),
        F.count(F.when(F.col("key_conflict"), 1)).alias(
            "quarantined_records"
        ),
        F.count(
            F.when(
                ~F.col("key_conflict") & ~F.col("record_eligible"),
                1,
            )
        ).alias("field_excluded_records"),
    ).first().asDict()

    # Check that no selected input disappeared or was counted twice.
    # Distinct rows must split into eligible, conflicting and field-excluded records.
    if (
        counts["input_records"] != input_records
        or counts["input_records"] != (
            counts["distinct_source_rows"] + counts["duplicate_extra_records"]
        )
        or counts["distinct_source_rows"] != (
            counts["eligible_records"]
            + counts["quarantined_records"]
            + counts["field_excluded_records"]
        )
    ):
        raise ValueError(f"{reference_name}: reference counts do not reconcile.")

    # Build the reference and audit from the resolved records whose totals we just checked.
    result = build_reference_outputs(resolved, reference_name, source_columns)

    # Count eligible output rows and how many have at least one quality flag.
    output_summary = result["reference"].agg(
        F.count("*").alias("output_records"),
        F.count(F.when(F.size("quality_flags") > 0, 1)).alias("flagged_records"),
    ).first().asDict()

    # Total the source records accounted for by each audit outcome.
    # Sum record_count because one audit entry can represent several identical copies.
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in result["audit"].groupBy("disposition").agg(
            F.sum("record_count").alias("source_records")
        ).collect()
    }
    # Require the audit to explain every input and its eligible total to match the reference table.
    if sum(dispositions.values()) != input_records:
        raise ValueError(f"{reference_name}: audit does not account for all inputs.")

    # Check that the eligible audit count matches the number of reference output rows.
    if dispositions.get("eligible", 0) != output_summary["output_records"]:
        raise ValueError(f"{reference_name}: eligible audit and output counts differ.")

    # Attach a small summary to the two Spark tables.
    result["summary"] = {
        "input_records": input_records,
        **output_summary,
        "dispositions": dispositions,
    }
    # Return the reference table, audit and summary together.
    return result


# Store each reference type's results, eligible table and audit for later use.
reference_results = {}
reference_tables = {}
reference_audits = {}

# Process the selected site, monitor and weather station records in turn.
for reference_name, _, _, _ in reference_specs:
    result = process_reference_records(
        records=reference_profile_rows[reference_name],
        source=reference_sources[reference_name],
        reference_name=reference_name,
        environment_root=context.environment_root,
    )

    # Save this reference type's outputs and take its summary for display.
    reference_results[reference_name] = result
    reference_tables[reference_name] = result["reference"]
    reference_audits[reference_name] = result["audit"]
    summary = result["summary"]

    # Total the audited source records and print the cleaning results.
    audited = sum(summary["dispositions"].values())
    print(f"{reference_name}: {summary['input_records']} selected, "
          f"{summary['output_records']} eligible ({summary['flagged_records']} flagged), "
          f"{audited} accounted for: {summary['dispositions']}")

# Optionally display the selected locations and their retained flags.
if show_diagnostics:
    for reference_name, reference_table in reference_tables.items():
        print(reference_name)
        reference_table.select(
            reference_output_specs[reference_name]["id_column"],
            *reference_type_specs[reference_name],
            "quality_flags",
        ).show(truncate=False, vertical=True)

epa_sites: 1 selected, 1 eligible (0 flagged), 1 accounted for: {'eligible': 1}
epa_monitors: 1 selected, 1 eligible (0 flagged), 1 accounted for: {'eligible': 1}
noaa_stations: 1 selected, 1 eligible (0 flagged), 1 accounted for: {'eligible': 1}


## 7. Publish source Silver

### 7.1 Define the eight output tables

Collect two measurement tables, three reference tables and three audits. Ozone observations have one eligible reading per monitor and UTC timestamp. Temperature reports keep one reading per station and original report time. Hourly selection happens later.

The clean tables use UTC timestamps, ozone in ppm and temperature in degrees Celsius. A contract defines each table's columns, types, required values and row keys.

Files stay in staging until validation passes. A manifest lists the files belonging to a published release and their checksums. A completion report records a successful reload.


In [26]:
def silver_column_types(data_type, names):
    """Return a column-to-type mapping for fields that share one Spark type."""
    return {name: data_type for name in names.split()}

# Shared lineage and audit fields.
silver_lineage_types = silver_column_types(
    "string",
    "source_provider source_dataset source_version_sha256 source_file_path "
    "source_file_sha256 source_manifest_path source_record_hash_scheme "
    "source_record_sha256",
)

silver_audit_types = {
    **silver_lineage_types,
    **silver_column_types("string", "field_disposition disposition"),
    **silver_column_types("array<string>", "field_reasons reasons"),
    "raw_fields": "map<string,string>",
    "field_flags": "map<string,boolean>",
    "key_variant_count": "bigint",
    "key_conflict": "boolean",
    "record_count": "bigint",
}

# Schemas for measurements, audits and reference records.
source_silver_schemas = {
    "ozone_observations": {
        **silver_lineage_types,
        **silver_column_types(
            "string",
            "state_code county_code site_number parameter_code poc monitor_id "
            "ozone_raw source_unit mdl_raw qualifier_code qualifier_raw "
            "method_type method_code method_name",
        ),
        "timestamp_utc": "timestamp",
        "ozone_ppm": "double",
        "mdl_ppm": "double",
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
    },
    "temperature_reports": {
        **silver_lineage_types,
        **silver_column_types(
            "string",
            "station_id temperature_raw timestamp_raw source_code quality_code "
            "report_type measurement_code source_station_id source_code_raw "
            "quality_code_raw report_type_raw measurement_code_raw "
            "source_station_id_raw",
        ),
        "timestamp_utc": "timestamp",
        "temperature_c": "double",
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
        "supporting_source_records": "bigint",
    },
    "ozone_audit": {
        **silver_audit_types,
        "monitor_id": "string",
        "timestamp_utc": "timestamp",
    },
    "temperature_audit": {
        **silver_audit_types,
        "station_id": "string",
        "timestamp_utc": "timestamp",
        "key_variant_count": "int",
        "key_source_records": "bigint",
        "redundant_temperature": "boolean",
    },
    "reference_audit": {
        **silver_audit_types,
        "reference": "string",
        "entity_id": "string",
        "reference_quality_flags": "array<string>",
    },
}

# Add the reference schemas from the field definitions above.
for reference_name, spec in reference_output_specs.items():
    source_silver_schemas[reference_name] = {
        **silver_lineage_types,
        **{name: "string" for name in spec["keys"].values()},
        spec["id_column"]: "string",
        **{
            name: data_type
            for name, (_, data_type) in reference_type_specs[reference_name].items()
        },
        **{name: "string" for name in spec["text"].values()},
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
    }

# Assemble the eight source Silver tables.
reference_audit_combined = reference_audits["epa_sites"].unionByName(
    reference_audits["epa_monitors"]
).unionByName(reference_audits["noaa_stations"])

source_silver_tables = {
    "ozone_observations": ozone_observations,
    "temperature_reports": temperature_observations,
    **reference_tables,
    "ozone_audit": ozone_audit,
    "temperature_audit": temperature_audit,
    "reference_audit": reference_audit_combined,
}

# Record the cleaning rules used for this bundle.
source_silver_rule_versions = {
    "ozone": "ozone-source-v1",
    "temperature": "temperature-source-v1",
    "references": "reference-source-v1",
}

def source_silver_contract(name, key, required, expected_rows=None):
    """Return the expected column types, unique key, required fields and optional row count."""
    return {
        "schema_version": f"{name}-v1",
        "columns": source_silver_schemas[name],
        "key": key,
        # Include keys and lineage once, preserving field order.
        "required": list(dict.fromkeys([*key, *required, *silver_lineage_types])),
        "expected_rows": expected_rows,
    }

# Set each table's key, required fields and expected row count.
source_silver_contracts = {
    "ozone_observations": source_silver_contract(
        "ozone_observations",
        ["monitor_id", "timestamp_utc"],
        "state_code county_code site_number parameter_code poc ozone_ppm "
        "source_unit quality_flags source_occurrences".split(),
        ozone_result_summary["after"]["records"],
    ),
    "temperature_reports": source_silver_contract(
        "temperature_reports",
        ["station_id", "timestamp_utc"],
        "temperature_c source_code quality_code report_type quality_flags "
        "source_occurrences supporting_source_records".split(),
        temperature_result_summary["after"]["records"],
    ),
}


# Expect one eligible reference row for each selected identifier.
for reference_name, spec in reference_output_specs.items():
    source_silver_contracts[reference_name] = source_silver_contract(
        reference_name,
        [spec["id_column"]],
        [
            *spec["keys"].values(), "latitude_deg", "longitude_deg",
            "quality_flags", "source_occurrences",
        ],
        expected_rows=1,
    )

# Source identity and disposition keep extra-copy audit entries distinct.
silver_audit_key = [
    "source_provider", "source_dataset", "source_version_sha256",
    "source_file_path", "source_file_sha256", "source_record_sha256",
    "disposition",
]
for dataset in ("ozone_audit", "temperature_audit", "reference_audit"):
    required = [
        "raw_fields", "field_flags", "field_disposition", "field_reasons",
        "key_conflict", "reasons", "record_count",
    ]
    # The shared audit also identifies the reference type.
    if dataset == "reference_audit":
        required += ["reference", "reference_quality_flags"]

    source_silver_contracts[dataset] = source_silver_contract(
        dataset, silver_audit_key.copy(), required,
    )

print(f"Source Silver contracts defined: {len(source_silver_contracts)} datasets.")


Source Silver contracts defined: 8 datasets.


### 7.2 Write the Parquet files

The regional outputs are small. The summary below shows each table’s row count, Parquet size and number of files. We write each table with one output partition to avoid producing several small files. Larger outputs would need more partitions to keep writes parallel.

Each table is written to this run’s staging folder, with its schema, row count, file sizes and checksums recorded for validation.


In [27]:
# Write an unpublished bundle and return the details needed for read-back validation.
def stage_source_silver(tables, contracts, context):
    """Write a new unpublished Parquet bundle and record its file inventory."""
    # Require every table to have a matching contract, with no missing or extra names.
    if set(tables) != set(contracts):
        raise ValueError("Tables and contracts must have the same dataset names.")

    # Keep staging inside this run's directory and check the table schemas before writing.
    root = Path(context.environment_root).resolve()
    parent = Path(context.staging_root).resolve()
    if parent == root / "staging" or not parent.is_relative_to(root / "staging"):
        raise ValueError("Staging must be inside this environment's run directory.")

    # Reject duplicate columns and schemas that differ from the contracts.
    for name, table in tables.items():
        if (
            len(table.columns) != len(set(table.columns))
            or dict(table.dtypes) != contracts[name]["columns"]
        ):
            raise ValueError(f"{name}: invalid dataset name or schema.")

    # Create a new staging folder so a previous attempt is never silently overwritten.
    stage_path = parent / f"source-silver-{time.time_ns()}"
    stage_path.mkdir(parents=True, exist_ok=False)
    
    started = time.perf_counter()
    created_at_utc = datetime.now(timezone.utc).isoformat()
    datasets = {}

    # Write each small regional table and record its row count.
    for name, table in tables.items():
        destination = stage_path / name
        # Counting and writing both run Spark jobs. Cache this table to reuse the result,
        # and release it afterwards only if this function created the cache.
        owned_cache = not table.is_cached
        if owned_cache:
            table.cache()
        try:
            # Count the rows so the saved table can later be checked against this total.
            row_count = table.count()
            # Write each small regional table with one partition.
            table.coalesce(1).write.parquet(
                str(destination), mode="errorifexists", compression="snappy",
            )
        # Release only the cache created here, even if counting or writing fails.
        finally:
            if owned_cache:
                table.unpersist(blocking=True)

        # Record each Parquet file's relative path, size and checksum after the write.
        files = [
            {
                "path": path.relative_to(stage_path).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": file_sha256(path),
            }
            for path in sorted(destination.glob("*.parquet"))
        ]
        # Stop if the write produced no Parquet files to validate.
        if not files:
            raise ValueError(f"{name}: no Parquet files were produced.")

        # Keep this table's row count, schema, file list and total size for later checks.
        datasets[name] = {
            "path": name,
            "row_count": row_count,
            "schema": table.schema.jsonValue(),
            "files": files,
            "bytes": sum(item["bytes"] for item in files),
        }
        print(f"Staged {name}: {row_count} rows, {len(files)} Parquet file(s).")

    # Use a consistent JSON format so dictionary key order does not change the checksum.
    contract_json = json.dumps(contracts, sort_keys=True, separators=(",", ":"))

    # Return the staging details for validation; the files are not published yet.
    return {
        "status": "staged",
        "created_at_utc": created_at_utc,
        "run_id": context.run_id,
        "environment": context.environment,
        "environment_root": str(root),
        "bundle_path": stage_path.relative_to(root).as_posix(),
        "format": "parquet",
        "compression": "snappy",
        "write_partitions": 1,
        "partition_columns": [],
        "contract_sha256": hashlib.sha256(contract_json.encode("utf-8")).hexdigest(),
        "datasets": datasets,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
    }

# Clear the previous result so a failed attempt cannot leave an old stage selected.
# Stage the actual source Silver tables now, then display the saved file inventory.
source_silver_stage = None
source_silver_stage = stage_source_silver(
    source_silver_tables, source_silver_contracts, context,
)

print("Status:", source_silver_stage["status"])
print("Staging directory:", (
    context.environment_root / source_silver_stage["bundle_path"]
))
print("Staging time (seconds):", source_silver_stage["elapsed_seconds"])

spark.createDataFrame(
    [
        (name, info["row_count"], len(info["files"]), info["bytes"])
        for name, info in source_silver_stage["datasets"].items()
    ],
    "dataset STRING, records LONG, parquet_files INT, parquet_bytes LONG",
).orderBy("dataset").show(truncate=False)

Staged ozone_observations: 5071 rows, 1 Parquet file(s).


Staged temperature_reports: 11686 rows, 1 Parquet file(s).
Staged epa_sites: 1 rows, 1 Parquet file(s).
Staged epa_monitors: 1 rows, 1 Parquet file(s).
Staged noaa_stations: 1 rows, 1 Parquet file(s).
Staged ozone_audit: 5166 rows, 1 Parquet file(s).


26/11/20 16:14:07 WARN DAGScheduler: Broadcasting large task binary with size 1280.6 KiB
26/11/20 16:14:16 WARN DAGScheduler: Broadcasting large task binary with size 1286.5 KiB
26/11/20 16:14:16 WARN DAGScheduler: Broadcasting large task binary with size 1487.9 KiB


Staged temperature_audit: 11851 rows, 1 Parquet file(s).


26/11/20 16:14:19 WARN DAGScheduler: Broadcasting large task binary with size 1008.7 KiB


Staged reference_audit: 3 rows, 1 Parquet file(s).
Status: staged
Staging directory: /home/ubuntu/dat535-final-project-data/dev/staging/0986e41d03894eabbfdfdacf001f3362/source-silver-1795191221358627849
Staging time (seconds): 38.712
+-------------------+-------+-------------+-------------+
|dataset            |records|parquet_files|parquet_bytes|
+-------------------+-------+-------------+-------------+
|epa_monitors       |1      |1            |10679        |
|epa_sites          |1      |1            |10124        |
|noaa_stations      |1      |1            |8249         |
|ozone_audit        |5166   |1            |444772       |
|ozone_observations |5071   |1            |385295       |
|reference_audit    |3      |1            |11478        |
|temperature_audit  |11851  |1            |3129147      |
|temperature_reports|11686  |1            |944606       |
+-------------------+-------+-------------+-------------+



### 7.3 Read back and check the saved tables

Reopen the staged Parquet files and check their hashes, schemas, counts, required values and keys. The clean outputs must match their audit totals and point to the selected Bronze sources.

Row hashes connect the clean observations to the accepted audit entries. In **8.4b**, we also compare the saved hourly tables with the prepared DataFrames.


In [28]:
# Reopen the staged tables and check their files, values, source details and audit totals.
def validate_source_silver(stage, contracts, context, inputs):
    """Validate staged files, table contracts, lineage and audit reconciliation."""
    # Require this stage, its contracts and the current run to agree.
    root = Path(context.environment_root).resolve()
    if (
        stage["status"] != "staged"
        or Path(stage["environment_root"]).resolve() != root
        or stage["environment"] != context.environment
        or stage["run_id"] != context.run_id
        or stage["contract_sha256"] != json_sha256(contracts)
        or set(stage["datasets"]) != set(contracts)
    ):
        raise ValueError("The staged bundle does not match this run and its contracts.")

    # Resolve the staged bundle without allowing paths outside this run's staging directory.
    bundle = child_path(root, stage["bundle_path"], must_exist=True)
    staging_root = Path(context.staging_root).resolve()
    if bundle == staging_root or not bundle.is_relative_to(staging_root):
        raise ValueError("The bundle is outside this run's staging directory.")

    # Read each saved dataset and collect its validation result.
    readback, checks = {}, {}
    verified_files = verify_file_inventory(bundle, stage["datasets"])
    for name, contract in contracts.items():
        info = stage["datasets"][name]
        paths = verified_files[name]

        # Read every listed Parquet file and fail if a file is missing or corrupt.
        table = (
            spark.read
            .option("mergeSchema", "true")
            .option("ignoreCorruptFiles", "false")
            .option("ignoreMissingFiles", "false")
            .parquet(*paths)
        )
        # Require the recorded column order and the data types defined by the contract.
        stored_columns = [field["name"] for field in info["schema"]["fields"]]
        if table.columns != stored_columns or dict(table.dtypes) != contract["columns"]:
            raise ValueError(f"{name}: persisted column names or types changed.")

        # Build missing-value checks for required fields, including blank required text.
        bad_required = F.lit(False)
        for column in contract["required"]:
            missing = F.col(column).isNull()
            # Required text must contain more than an empty string or spaces.
            if contract["columns"][column] == "string":
                missing = missing | (F.trim(F.col(column)) == "")
            bad_required = bad_required | missing

        # Check finite numbers, array contents, hash formats and valid positive counts.
        bad_numeric, bad_array, bad_hash, bad_count = (F.lit(False) for _ in range(4))
        for column, data_type in contract["columns"].items():
            value = F.col(column)
            # Flag numeric values that are NaN or infinity.
            if data_type in {"float", "double"}:
                bad_numeric = bad_numeric | (
                    value.isNotNull() & ~finite_number(column)
                )
            # Flag arrays containing null entries.
            if data_type == "array<string>":
                bad_array = bad_array | F.coalesce(
                    F.exists(value, lambda item: item.isNull()), F.lit(False),
                )
            # Require a 64-character checksum containing only digits and lowercase a-f.
            if column.endswith("_sha256"):
                bad_hash = bad_hash | ~F.coalesce(
                    value.rlike("^[0-9a-f]{64}$"), F.lit(False),
                )
        # Check the available count columns; any non-null count must be at least one.
        for column in (
            "source_occurrences", "supporting_source_records",
            "record_count", "key_variant_count", "key_source_records",
        ):
            if column in table.columns:
                bad_count = bad_count | (
                    F.col(column).isNotNull() & (F.col(column) < 1)
                )
        # Each audit entry counts one distinct record unless it represents extra copies.
        if "record_count" in table.columns:
            bad_count = bad_count | (
                (F.col("disposition") != "exact_duplicate")
                & (F.col("record_count") != 1)
            )
        # Name the checks so their failure counts can explain an unsuccessful validation.
        failures = {
            "missing_required": bad_required, "nonfinite_numbers": bad_numeric,
            "null_array_items": bad_array, "invalid_hashes": bad_hash,
            "invalid_counts": bad_count,
        }
        # Make Spark read every column, including nested audit values.
        # Counting rows alone would not require Spark to decode all those fields.
        decoded = F.to_json(
            F.struct(*[F.col(column) for column in table.columns]),
            options={"ignoreNullFields": "false"},
        )
        # Run the checks in Spark and collect only the totals.
        measured = table.agg(
            F.count("*").alias("records"),
            F.sum(F.length(decoded)).alias("decoded_characters"),
            *[
                F.count(F.when(condition, 1)).alias(label)
                for label, condition in failures.items()
            ],
        ).first().asDict()

        # Stop if any saved row fails a value check.
        if any(measured[label] for label in failures):
            raise ValueError(f"{name}: persisted value checks failed: {measured}")

        # Match the staging row count and any specific row count required by the contract.
        if (
            measured["records"] != info["row_count"]
            or (
                contract["expected_rows"] is not None
                and measured["records"] != contract["expected_rows"]
            )
        ):
            raise ValueError(f"{name}: persisted row count does not match.")

        # Require each persisted table key to identify at most one row.
        if table.groupBy(*contract["key"]).count().filter("count > 1").limit(1).count():
            raise ValueError(f"{name}: duplicate output keys.")

        # Keep the reloaded table and its successful check results for the audit comparisons.
        readback[name] = table
        checks[name] = {
            "records": measured["records"], "parquet_files": len(paths),
            "status": "passed",
        }

    # Ensure every saved table is covered as either a clean output or its audit.
    audit_names = {item["audit"] for item in inputs.values()}
    if (set(inputs) & audit_names) or (set(inputs) | audit_names) != set(readback):
        raise ValueError("Every persisted dataset must have one validation role.")

    audit_checks, accounted_audit_rows = {}, Counter()
    lineage_columns = [*silver_lineage_types]
    # Match each clean table to its audit and selected Bronze source.
    for name, expected in inputs.items():
        clean = readback[name]
        audit_name = expected["audit"]
        audit = readback[audit_name]
        # The reference audit is shared, so select only entries for this reference type.
        if "reference" in expected:
            audit = audit.filter(F.col("reference") == F.lit(expected["reference"]))

        # Compare represented input counts by outcome, not just the number of audit entries.
        audit_rows = audit.groupBy("disposition").agg(
            F.sum("record_count").alias("source_records"),
            F.count("*").alias("audit_rows"),
        ).collect()
        dispositions = {row["disposition"]: row["source_records"] for row in audit_rows}

        # Track audit entries separately from the number of source records they represent.
        accounted_audit_rows[audit_name] += sum(row["audit_rows"] for row in audit_rows)
        if (
            dispositions != expected["dispositions"]
            or sum(dispositions.values()) != expected["input_records"]
            or dispositions.get("eligible", 0) != checks[name]["records"]
        ):
            raise ValueError(f"{name}: persisted audit counts do not reconcile.")

        # Compare quality-flag totals with the processing summary.
        if "quality_flags" in expected:
            flags = {
                row["quality_flag"]: row["count"]
                for row in clean.select(
                    F.explode("quality_flags").alias("quality_flag")
                ).groupBy("quality_flag").count().collect()
            }
            if flags != expected["quality_flags"]:
                raise ValueError(f"{name}: persisted quality-flag counts differ.")
        # For references, compare the number of rows with at least one quality flag.
        if "flagged_records" in expected:
            if clean.filter(F.size("quality_flags") > 0).count() != expected["flagged_records"]:
                raise ValueError(f"{name}: persisted flagged-record count differs.")

        # Require clean and audit rows to point to the selected Bronze file and version.
        # eqNullSafe treats two nulls as equal and a null versus a value as different.
        lineage = selected_bronze_lineage(expected["source"], root)
        mismatch = F.lit(False)
        for column, value in lineage.items():
            mismatch = mismatch | ~F.col(column).eqNullSafe(F.lit(value))
        combined_lineage = clean.select(*lineage).unionByName(audit.select(*lineage))
        if combined_lineage.filter(mismatch).limit(1).count():
            raise ValueError(f"{name}: lineage differs from the selected Bronze source.")

        # Find missing and extra source-row identities in both directions.
        # exceptAll preserves copy counts, so repeated rows cannot hide a mismatch.
        accepted = audit.filter(F.col("disposition") == "eligible").select(*lineage_columns)
        observed = clean.select(*lineage_columns)
        differences = observed.exceptAll(accepted).unionByName(accepted.exceptAll(observed))
        if differences.limit(1).count():
            raise ValueError(f"{name}: output and eligible-audit lineage differ.")

        # Record the passed audit totals and matching Bronze source details.
        audit_checks[name] = {
            "input_records": expected["input_records"],
            "dispositions": dispositions,
            "lineage": lineage,
            "status": "passed",
        }

    # Check that the validation above covered every row of each audit exactly once.
    if any(accounted_audit_rows[name] != checks[name]["records"] for name in audit_names):
        raise ValueError("Some audit rows are unaccounted for or counted more than once.")

    # Return validation evidence tied to this exact stage and the reloaded Spark tables.
    report = {
        "status": "validated",
        "validated_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_id": context.run_id,
        "bundle_path": stage["bundle_path"],
        "stage_sha256": json_sha256(stage),
        "contract_sha256": stage["contract_sha256"],
        "datasets": checks,
        "audits": audit_checks,
    }
    return report, readback


# Supply the expected input counts, audit outcomes and flags from the processing summaries.
source_silver_validation_inputs = {
    "ozone_observations": {
        "source": ozone_source, "audit": "ozone_audit",
        "input_records": ozone_result_summary["before"]["records"],
        "dispositions": ozone_result_summary["dispositions"],
        "quality_flags": ozone_result_summary["quality_flags"],
    },
    "temperature_reports": {
        "source": temperature_source, "audit": "temperature_audit",
        "input_records": temperature_result_summary["before"]["records"],
        "dispositions": temperature_result_summary["dispositions"],
        "quality_flags": temperature_result_summary["quality_flags"],
    },
}

# Add each reference type's expected totals and identify its part of the shared audit.
for reference_name, result in reference_results.items():
    source_silver_validation_inputs[reference_name] = {
        "source": reference_sources[reference_name],
        "audit": "reference_audit", "reference": reference_name,
        "input_records": result["summary"]["input_records"],
        "dispositions": result["summary"]["dispositions"],
        "flagged_records": result["summary"]["flagged_records"],
    }

# Clear earlier results so a failed check cannot leave old validation results selected.
source_silver_validation = None
source_silver_readback = None
source_silver_validation, source_silver_readback = validate_source_silver(
    source_silver_stage, source_silver_contracts, context,
    source_silver_validation_inputs,
)

spark.createDataFrame(
    [
        (name, result["records"], result["parquet_files"], result["status"])
        for name, result in source_silver_validation["datasets"].items()
    ],
    "dataset STRING, records LONG, parquet_files INT, validation STRING",
).orderBy("dataset").show(truncate=False)

print("Audit and lineage reconciliations passed:", len(source_silver_validation["audits"]))
print("Validation status:", source_silver_validation["status"])
print("Publication status: unpublished")

+-------------------+-------+-------------+----------+
|dataset            |records|parquet_files|validation|
+-------------------+-------+-------------+----------+
|epa_monitors       |1      |1            |passed    |
|epa_sites          |1      |1            |passed    |
|noaa_stations      |1      |1            |passed    |
|ozone_audit        |5166   |1            |passed    |
|ozone_observations |5071   |1            |passed    |
|reference_audit    |3      |1            |passed    |
|temperature_audit  |11851  |1            |passed    |
|temperature_reports|11686  |1            |passed    |
+-------------------+-------+-------------+----------+

Audit and lineage reconciliations passed: 5
Validation status: validated
Publication status: unpublished


### 7.4 Define how a release is published

After validation passes, move the files to the published folder and write the manifest last. Later stages use the manifest to find the completed release.

This cell defines the publisher. The following test cases call it on isolated copies before we publish the actual data. The same publication attempt can reuse its saved release when the metadata and files still match.


In [29]:
def publish_source_silver(
    stage, validation, contracts, context, *, profile, rule_versions, provenance,
):
    """Move validated source tables to their release directory and write the manifest last."""
    # Make an independent JSON copy so the manifest records the supplied metadata consistently.
    stage, validation, contracts, profile, rule_versions, provenance = json.loads(
        json.dumps(
            [stage, validation, contracts, profile, rule_versions, provenance],
            allow_nan=False,
        )
    )
    # Resolve the environment folder and calculate checksums for the stage details and contracts.
    root = Path(context.environment_root).resolve()
    stage_hash = json_sha256(stage)
    contract_hash = json_sha256(contracts)

    # Require the validation to match this exact run, staged bundle and set of contracts.
    if (
        stage["status"] != "staged"
        or Path(stage["environment_root"]).resolve() != root
        or stage["environment"] != context.environment
        or stage["run_id"] != context.run_id
        or stage["contract_sha256"] != contract_hash
        or validation["status"] != "validated"
        or validation["run_id"] != context.run_id
        or validation["bundle_path"] != stage["bundle_path"]
        or validation["stage_sha256"] != stage_hash
        or validation["contract_sha256"] != contract_hash
        or set(stage["datasets"]) != set(contracts)
        or set(validation["datasets"]) != set(contracts)
    ):
        raise ValueError(
            "Publication requires a matching stage, validation and contracts."
        )

    # Require a passed file and count check for every table.
    for name, info in stage["datasets"].items():
        check = validation["datasets"][name]
        if (
            check["status"] != "passed"
            or check["records"] != info["row_count"]
            or check["parquet_files"] != len(info["files"])
        ):
            raise ValueError(
                f"{name}: validation is incomplete or inconsistent."
            )

    # All five clean tables must also have passed their audit and source-tracing checks.
    clean_names = {
        "ozone_observations", "temperature_reports",
        "epa_sites", "epa_monitors", "noaa_stations",
    }
    if (
        set(validation["audits"]) != clean_names
        or any(
            item["status"] != "passed"
            for item in validation["audits"].values()
        )
    ):
        raise ValueError(
            "All five audit and lineage reconciliations must pass."
        )

    # Use the profile region and year in the release path.
    region = profile["region"].strip().lower().replace(" ", "_")
    year = profile["year"]

    # Resolve and check the staging location before moving any files.
    staged_bundle = child_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if (
        not staging_root.is_relative_to(root / "staging")
        or staged_bundle == staging_root
        or not staged_bundle.is_relative_to(staging_root)
    ):
        raise ValueError(
            "The bundle is outside this run's staging directory."
        )

    # Use the stage hash to identify this publication attempt.
    # It includes run metadata, so a fresh run can get a new ID with the same table values.
    publication_id = stage_hash

    # Build separate locations for the saved data and its publication manifest.
    suffix = f"silver/source/{region}/{year}/{publication_id}"
    final_bundle = child_path(root, f"data/{suffix}")
    manifest_path = child_path(
        root, f"published/{suffix}/manifest.json"
    )
    publication_root = child_path(root, "published/silver/source")

    # Save the table contracts, input profile, rule versions and provenance in the manifest.
    identity = {
        "manifest_version": 1,
        "kind": "source_silver",
        "status": "published",
        "publication_id": publication_id,
        "run_id": context.run_id,
        "environment": context.environment,
        "bundle_path": final_bundle.relative_to(root).as_posix(),
        "stage_sha256": stage_hash,
        "contract_sha256": contract_hash,
        "profile": profile,
        "rule_versions": rule_versions,
        "provenance": provenance,
        "contracts": contracts,
        "storage": {
            key: stage[key]
            for key in (
                "format", "compression", "write_partitions",
                "partition_columns",
            )
        },
        "datasets": stage["datasets"],
    }

    # Allow only one source Silver publisher to write in this environment at a time.
    publication_root.mkdir(parents=True, exist_ok=True)
    lock = publication_root / ".publish.lock"
    try:
        lock.mkdir()
    # An existing lock may belong to an active or interrupted attempt, so stop.
    except FileExistsError as error:
        raise RuntimeError(
            "Publication lock already exists; inspect it before retrying: "
            f"{lock}"
        ) from error

    try:
        # An identical retry reuses the release only if its metadata and stored files still match.
        if manifest_path.exists():
            existing = json.loads(
                manifest_path.read_text(encoding="utf-8")
            )
            # Require the existing release to match this attempt's recorded settings and file details.
            if {key: existing.get(key) for key in identity} != identity:
                raise ValueError(
                    "The existing publication has different metadata."
                )

            # Compare the supplied validation results with the saved ones, ignoring the check time.
            recorded_validation = existing.get("validation", {})
            if any(
                recorded_validation.get(key) != value
                for key, value in validation.items()
                if key != "validated_at_utc"
            ):
                raise ValueError(
                    "The existing publication has different validation results."
                )

            # Recheck the published Parquet files before reusing the existing release.
            verify_file_inventory(
                final_bundle, existing["datasets"]
            )
            
            return {
                "outcome": "reused",
                "manifest_path": str(manifest_path),
                "manifest": existing,
            }

        # Data without a manifest is an incomplete attempt; do not overwrite it.
        if final_bundle.exists() or final_bundle.is_symlink():
            raise FileExistsError(
                f"Data exists without a publication manifest: {final_bundle}. "
                "Inspect the incomplete attempt, then stage and validate "
                "a new attempt."
            )

        # Verify the staged files once more before moving them to the release directory.
        verify_file_inventory(
            staged_bundle, stage["datasets"]
        )

        final_bundle.parent.mkdir(parents=True, exist_ok=True)
        manifest_path.parent.mkdir(parents=True, exist_ok=True)

        # Add the publication time and validation evidence to the release manifest.
        manifest = {
            **identity,
            "published_at_utc": datetime.now(timezone.utc).isoformat(),
            "validation": validation,
        }

        # Readers may use the bundle only after its manifest exists.
        staged_bundle.rename(final_bundle)
        write_json(manifest_path, manifest)

        return {
            "outcome": "published",
            "manifest_path": str(manifest_path),
            "manifest": manifest,
        }
    # Release the publication lock whether publication succeeds or raises an error.
    finally:
        lock.rmdir()

print("Source Silver publication helper ready. No data published.")

Source Silver publication helper ready. No data published.


### 7.5 Test the cleaning and publication rules

Small cases in `tests/silver_checks.py` check the readers, cleaning rules, reference records and publication. They include cases missing from the Alabama inputs, such as conflicting readings.

Publication tests use copies under staging to check successful runs, retries and failures. The results are recorded with the run ID and a checksum of the stage description, so publication can confirm that the matching checks passed.


In [30]:
# Run the reader, cleaning and publication test cases.
import silver_checks

fixture_results = {
    "readers": silver_checks.verify_delimited_readers(
        staging_root=context.staging_root,
        read_ozone_csv=partial(read_source, spark, delimiter=","),
        read_delimited_file=partial(read_source, spark),
    ),
    "ozone": silver_checks.verify_ozone_rules(
        spark=spark,
        type_ozone_fields=type_ozone_fields,
        ozone_field_checks=ozone_field_checks,
        classify_ozone_fields=classify_ozone_fields,
        resolve_source_keys=resolve_source_keys,
        ozone_key_columns=ozone_key_columns,
        build_ozone_observations=build_ozone_observations,
        build_ozone_audit=build_ozone_audit,
    ),
    "temperature": silver_checks.verify_temperature_rules(
        spark=spark,
        profile=alabama_profile,
        temperature_columns=temperature_inspection_columns,
        type_temperature_fields=type_temperature_fields,
        temperature_field_checks=temperature_field_checks,
        classify_temperature_fields=classify_temperature_fields,
        resolve_temperature_keys=resolve_temperature_keys,
        build_temperature_observations=build_temperature_observations,
        build_temperature_audit=build_temperature_audit,
    ),
    "reference_fields": silver_checks.verify_reference_fields(
        spark=spark,
        type_reference_fields=type_reference_fields,
        classify_reference_fields=classify_reference_fields,
    ),
    "reference_outputs": silver_checks.verify_reference_outputs(
        spark=spark,
        type_reference_fields=type_reference_fields,
        classify_reference_fields=classify_reference_fields,
        resolve_reference_keys=resolve_reference_keys,
        build_reference_outputs=build_reference_outputs,
    ),
    # Test publication, retries and failures using isolated copies under staging.
    # Check that failed attempts preserve the previous successful publication.
    "publication": silver_checks.verify_silver_publication(
        stage=source_silver_stage,
        validation=source_silver_validation,
        contracts=source_silver_contracts,
        context=context,
        profile=alabama_profile,
        rule_versions=source_silver_rule_versions,
        publish_source_silver=publish_source_silver,
        verify_silver_file_inventory=verify_file_inventory,
        silver_json_sha256=json_sha256,
    ),
}
print(json.dumps(fixture_results, indent=2))

# Record passing test results for this run and staged bundle.
source_silver_checks = {
    "status": "passed",
    "run_id": context.run_id,
    "stage_sha256": json_sha256(source_silver_stage),
    "results": fixture_results,
}

26/11/20 16:14:32 WARN TaskSetManager: Lost task 0.0 in stage 567.0 (TID 375) (192.168.2.254 executor 0): org.apache.spark.SparkException: Encountered error while reading file file:///home/ubuntu/dat535-final-project-data/dev/staging/0986e41d03894eabbfdfdacf001f3362/readers-1795191271782547235/ozone_csv-too_few.csv. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:863)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:293)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$1(SparkPlan.scala:388)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:890)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:890)
	at org.apache.spark.rdd.MapPartitionsRDD.c

PASS: ozone_csv: decoded values and native width errors (6 cases)


26/11/20 16:14:33 WARN TaskSetManager: Lost task 0.0 in stage 573.0 (TID 387) (192.168.2.254 executor 0): org.apache.spark.SparkException: Encountered error while reading file file:///home/ubuntu/dat535-final-project-data/dev/staging/0986e41d03894eabbfdfdacf001f3362/readers-1795191271782547235/temperature_psv-too_few.psv. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:863)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:293)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$1(SparkPlan.scala:388)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:890)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:890)
	at org.apache.spark.rdd.MapPartition

PASS: temperature_psv: decoded values and native width errors (6 cases)
Ozone field fixtures passed: 17
Ozone key fixtures passed: 12
Ozone audit fixtures passed: 24 audit entries account for 25 source records.
Ozone observation fixtures passed: 19 cases, 16 output records.
Temperature field fixtures passed: 42 cases.
Temperature key fixtures passed: 18 cases, 38 source records.
Temperature observation fixtures passed: 26 cases, 16 output records.
Temperature audit fixtures passed: 45 audit entries account for 46 source records.
epa_sites: field fixtures passed: 27
epa_monitors: field fixtures passed: 26
noaa_stations: field fixtures passed: 21
epa_sites: key, output and audit fixtures passed: 13
epa_monitors: key, output and audit fixtures passed: 13
noaa_stations: key, output and audit fixtures passed: 13
successful_publication: passed
identical_attempt_reused: passed
failed_validation: passed
changed_receipt: passed
changed_file: passed
missing_file: passed
unlisted_file_ignored: pa

### 7.6 Publish this run

Publish the validated tables with their source versions, profile, cleaning rules and test results.

The reports record the settings, runtime, Git revision and whether there are uncommitted changes. The revision identifies the commit; the flag shows whether the working copy differs from it.

Save edits and restart the kernel before a fresh run so it uses the saved notebook and modules.


In [31]:
# Publish the actual source Silver bundle after its fixture checks pass.

# Clear the previous publication result before starting this attempt.
source_silver_publication = None

# Require passing tests for this exact run and staged bundle before publishing.
if (
    source_silver_checks is None
    or source_silver_checks["run_id"] != context.run_id
    or source_silver_checks["stage_sha256"]
    != json_sha256(source_silver_stage)
    or source_silver_checks["status"] != "passed"
):
    raise RuntimeError(
        "Run 7.5 for this staged bundle before publishing."
    )

# Keep run provenance and the results of the fixtures just executed.
source_silver_provenance = {**run_provenance, "fixture_checks": source_silver_checks}

# Move the validated bundle and write its publication manifest.
source_silver_publication = publish_source_silver(
    source_silver_stage,
    source_silver_validation,
    source_silver_contracts,
    context,
    profile=alabama_profile,
    rule_versions=source_silver_rule_versions,
    provenance=source_silver_provenance,
)
# Keep the manifest checksum so the next loader can verify this exact release.
source_silver_publication["manifest_sha256"] = file_sha256(
    Path(source_silver_publication["manifest_path"])
)

# Clear the staged readback; the next step reloads the published files.
source_silver_readback = None

print("Publication outcome:", source_silver_publication["outcome"])
print(
    "Publication status:",
    source_silver_publication["manifest"]["status"],
)
print(
    "Published datasets:",
    len(source_silver_publication["manifest"]["datasets"]),
)
print("Manifest:", source_silver_publication["manifest_path"])
print(
    "Manifest SHA-256:",
    source_silver_publication["manifest_sha256"],
)
print("Fixture status:", source_silver_checks["status"])

Publication outcome: published
Publication status: published
Published datasets: 8
Manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/source/alabama/2025/1b4389e89c9a616e54b11e42a7941a569c8d828e0e31a30843d96b16139c2709/manifest.json
Manifest SHA-256: 4247f78be85f04fa04a401fe41fc1bc738359ae76e48414cd71bdfc617fd45e0
Fixture status: passed


### 7.7 Reload source Silver

We use the manifest to reopen the eight tables and verify their files and counts. The hourly preparation below starts from this saved release.

In [32]:
# Clear reload results so a failure cannot leave stale tables selected.
source_silver_loaded_manifest = None
source_silver_loaded_tables = None
source_silver_loaded_counts = None

# Verify and reopen the source release.
(
    source_silver_loaded_manifest,
    source_silver_loaded_tables,
    source_silver_loaded_counts,
) = load_source_silver(
    source_silver_publication["manifest_path"],
    context,
    spark=spark,
    expected_manifest_sha256=(
        source_silver_publication["manifest_sha256"]
    ),
    profile=alabama_profile,
    rule_versions=source_silver_rule_versions,
)

spark.createDataFrame(
    sorted(source_silver_loaded_counts.items()),
    "dataset STRING, records LONG",
).show(truncate=False)

# Record this successful reload and its exact manifest in the receipt.
source_silver_completion = {
    "status": "success",
    "stage": "source_silver",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": context.run_id,
    "environment": context.environment,
    "manifest_path": (
        Path(source_silver_publication["manifest_path"])
        .relative_to(context.environment_root)
        .as_posix()
    ),
    "manifest_sha256": source_silver_publication["manifest_sha256"],
    "publication_outcome": source_silver_publication["outcome"],
    "datasets": source_silver_loaded_counts,
    "fixture_status": source_silver_checks["status"],
    "fixture_results": source_silver_checks["results"],
}
source_silver_completion_path = (
    context.reports_dir / "source_silver_completion.json"
)
write_json(
    source_silver_completion_path, source_silver_completion
)

print(
    "Source Silver loaded and verified:",
    len(source_silver_loaded_tables),
    "datasets.",
)
print("Source Silver status:", source_silver_completion["status"])
print("Fixture status:", source_silver_checks["status"])
print(
    "Profile status:",
    source_silver_loaded_manifest["profile"]["status"],
)
print("Completion report:", source_silver_completion_path)
# Downstream stages use saved tables; release the source caches.
ozone_input.unpersist(blocking=True)
temperature_input.unpersist(blocking=True)
for records in reference_profile_rows.values():
    records.unpersist(blocking=True)
print("Source caches released.")


+-------------------+-------+
|dataset            |records|
+-------------------+-------+
|epa_monitors       |1      |
|epa_sites          |1      |
|noaa_stations      |1      |
|ozone_audit        |5166   |
|ozone_observations |5071   |
|reference_audit    |3      |
|temperature_audit  |11851  |
|temperature_reports|11686  |
+-------------------+-------+

Source Silver loaded and verified: 8 datasets.
Source Silver status: success
Fixture status: passed
Profile status: provisional
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/0986e41d03894eabbfdfdacf001f3362/reports/source_silver_completion.json
Source caches released.


## 8. Silver: match ozone and temperature by hour

Hourly temperature contains one selected report per station and UTC hour. We choose the latest eligible report within that hour and join it to ozone.

Ozone readings stay in the output when temperature is missing. Checks on the keys and row counts make sure the join has not duplicated any ozone rows.

A full-year calendar records observed, excluded and missing hours, so coverage remains visible throughout the year. The result contains six unpublished tables and their preparation checks.


In [33]:
from silver_hourly import (
    build_yearly_mapping,
    enrich_hourly_ozone,
    prepare_hourly_silver,
    select_hourly_temperature,
)

# Use the source Silver release reloaded above.
hourly_manifest = source_silver_loaded_manifest

# Source inputs for the hourly preparation.
source_tables_for_hourly = source_silver_loaded_tables

# Use the profile recorded in this published source Silver release.
hourly_profile = hourly_manifest["profile"]

# Prepare the six hourly tables and their checks.
prepared_silver = prepare_hourly_silver(
    source_tables_for_hourly, hourly_manifest,
    environment=context.environment,
)

# Display coverage and checks. These prepared tables have not been published yet.
print("Preparation status:", prepared_silver["status"])
print("In-memory checks:", prepared_silver["checks"]["status"])
print("Region and year:", prepared_silver["profile"]["region"], prepared_silver["profile"]["year"])
print("Eligible ozone hours:", prepared_silver["checks"]["coverage"]["eligible_ozone_hours"])
print("Eligible temperature hours:", prepared_silver["checks"]["coverage"]["eligible_temperature_hours"])
print("Shared hours:", prepared_silver["checks"]["coverage"]["shared_hours"])
print("Calendar hours:", prepared_silver["checks"]["coverage"]["calendar_hours"])
print("Months:", prepared_silver["checks"]["monthly"]["months"])
print("Mapping approval:", prepared_silver["mapping_evidence"]["approval_status"])

Preparation status: prepared_unpublished
In-memory checks: passed
Region and year: Alabama 2025
Eligible ozone hours: 5071
Eligible temperature hours: 8514
Shared hours: 4836
Calendar hours: 8760
Months: 12
Mapping approval: pending


### 8.1 Inspect the hourly match and coverage

The first example shows which weather report was chosen in an hour with several candidates. The next two show an ozone hour with missing temperature and a month with low ozone coverage.

In [34]:
# Inspect three examples that explain the hourly selection and coverage.
prepared_hourly_tables = prepared_silver["tables"]
# Find the first hour with more than one eligible temperature report.
choice = (
    prepared_hourly_tables["temperature_hourly"].where(F.col("qualifying_report_count") > 1)
    .orderBy("hour_start_utc")
    .select("station_id", "hour_start_utc", "report_timestamp_utc")
    .limit(1)
)
# Show its candidate reports and mark the report selected for that hour.
candidates = (
    source_tables_for_hourly["temperature_reports"]
    .withColumn("hour_start_utc", F.date_trunc("hour", "timestamp_utc"))
    .join(choice, ["station_id", "hour_start_utc"])
    .select(
        "timestamp_utc", "temperature_c",
        (F.col("timestamp_utc") == F.col("report_timestamp_utc")).alias("selected"),
    )
)
print("Eligible reports in the first multi-report hour (empty if none):")
candidates.orderBy("timestamp_utc").show(truncate=False)

# Show one ozone hour retained with missing weather.
prepared_hourly_tables["ozone_enriched"].where(~F.col("temperature_available")).select(
    "monitor_id", "hour_start_utc", "ozone_ppm", "temperature_c", "integration_flags"
).orderBy("hour_start_utc").show(1, truncate=False)
# Show the month with the lowest eligible ozone coverage, including months with none.
prepared_hourly_tables["monthly_coverage"].select(
    "month_start_utc", "calendar_hours", "eligible_ozone_hours", "shared_hours", "ozone_calendar_pct"
).orderBy("ozone_calendar_pct", "month_start_utc").show(1, truncate=False)

Eligible reports in the first multi-report hour (empty if none):
+-------------------+-------------+--------+
|timestamp_utc      |temperature_c|selected|
+-------------------+-------------+--------+
|2025-01-05 15:15:00|9.4          |false   |
|2025-01-05 15:53:00|8.9          |true    |
+-------------------+-------------+--------+

+-------------------+-------------------+---------+-------------+-------------------------+
|monitor_id         |hour_start_utc     |ozone_ppm|temperature_c|integration_flags        |
+-------------------+-------------------+---------+-------------+-------------------------+
|01-073-1005-44201-1|2025-06-01 14:00:00|0.04     |NULL         |[no_eligible_temperature]|
+-------------------+-------------------+---------+-------------+-------------------------+
only showing top 1 row

+-------------------+--------------+--------------------+------------+------------------+
|month_start_utc    |calendar_hours|eligible_ozone_hours|shared_hours|ozone_calendar_pct|


### 8.2 Test hourly integration

The test cases check the selected report, an ozone hour with missing weather, and a leap-year calendar.

In [35]:
# Run small cases for hourly selection, missing weather and calendar coverage.
hourly_fixture_result = silver_checks.verify_hourly_integration(
    spark=spark,
    profile=hourly_profile,
    references=source_tables_for_hourly,
    build_yearly_mapping=build_yearly_mapping,
    select_hourly_temperature=select_hourly_temperature,
    enrich_hourly_ozone=enrich_hourly_ozone,
    prepare_hourly_silver=prepare_hourly_silver,
)
# Keep the fixture result as evidence for the hourly publication.
hourly_fixture_checks = {
    "status": "passed",
    "results": hourly_fixture_result,
}
print("Hourly integration fixtures passed:", hourly_fixture_result)

Hourly integration fixtures passed: {'checks': 29, 'calendar_hours': 8784, 'months': 12}


### 8.3 How well does the Alabama pairing cover 2025?

McAdory had eligible ozone readings in **5,071 of 8,760 calendar hours (57.89%)**. Of those 5,071 hours, **4,836 (95.37%)** had eligible temperature. The other 235 ozone readings stay in the dataset with a missing temperature.

The following cell records `accepted_with_limitations` for McAdory and Alabaster after the preparation and test checks pass. The decision is linked to the profile, source release, rules and mapping.

The weather station is about 26.7 km away, and historical operating schedules are unknown. These results describe the available observations for this pair. Gaps do not tell us why readings are absent.


In [36]:
# Clear the previous decision and approval before checking this mapping.
alabama_silver_acceptance = None

decision_mapping = prepared_silver["mapping_evidence"]
decision_mapping["approval_status"] = "pending"
decision_mapping.pop("approval_decision_sha256", None)

# Restrict this decision to the reviewed McAdory-Alabaster pair in 2025.
if (
    hourly_profile["region"] != "Alabama"
    or hourly_profile["year"] != 2025
    or prepared_silver["mapping_evidence"]["monitor_id"]
    != "01-073-1005-44201-1"
    or prepared_silver["mapping_evidence"]["station_id"]
    != "USW00053864"
):
    raise ValueError(
        "This acceptance decision applies to the reviewed Alabama pairing."
    )

# Require preparation and fixture checks to pass before recording acceptance.
if prepared_silver["checks"]["status"] != "passed":
    raise ValueError("In-memory preparation checks must pass.")

if hourly_fixture_checks["status"] != "passed":
    raise ValueError(
        "Run the hourly integration fixtures before recording acceptance."
    )

# Unresolved mapping flags need review; the code does not approve them automatically.
decision_mapping_flags = sorted(set(decision_mapping["review_flags"]))
if decision_mapping_flags:
    raise ValueError(
        "Mapping metadata requires review before acceptance: "
        + ", ".join(decision_mapping_flags)
    )

# Save the observed coverage and limitations with the profile, source and rules that were reviewed.
decision_coverage = prepared_silver["checks"]["coverage"]
alabama_silver_acceptance = {
    "decision_version": "available-observations-v1",
    "status": "accepted_with_limitations",
    "region": hourly_profile["region"],
    "year": hourly_profile["year"],
    "profile_sha256": json_sha256(prepared_silver["profile"]),
    "source_silver_publication_id": (
        prepared_silver["source_silver_publication_id"]
    ),
    "mapping_id": prepared_silver["mapping_evidence"]["mapping_id"],
    "mapping_review_flags": decision_mapping_flags,
    "rules_sha256": json_sha256(prepared_silver["rules"]),
    "coverage": {
        name: decision_coverage[name]
        for name in (
            "calendar_hours",
            "eligible_ozone_hours",
            "eligible_temperature_hours",
            "shared_hours",
        )
    },
    "critical_failures": [
        "Incompatible or unverifiable source publication.",
        "Missing, ambiguous or invalid required mapping.",
        "Mapping metadata flags requiring review.",
        "Unusable or repeated output keys, or row multiplication.",
        "Unreconciled source audits, calendar counts or gap intervals.",
        "Failed integration fixtures or publication read-back checks.",
    ],
    "reportable_limitations": [
        "Missing and excluded observations remain visible across all twelve months.",
        "Missing temperature remains null; eligible ozone is retained.",
        "Historical operating schedules are unknown.",
        (
            "Monitor-station distance: "
            f"{prepared_silver['mapping_evidence']['distance_km']:.2f} km."
        ),
        "Results describe the selected monitor-station pair and available observations.",
        "Observed gaps do not establish seasonal operation or instrument downtime.",
    ],
    # Record that publication still depends on the remaining checks,
    # including reading the saved output files back and validating them.
    "publication_condition": (
        "All publication and read-back checks must pass."
    ),
}

# Write the decision separately from the provisional profile JSON.
alabama_acceptance_path = (
    context.reports_dir / "alabama_silver_acceptance.json"
)
write_json(alabama_acceptance_path, alabama_silver_acceptance)

# Include the decision checksum so later checks can detect a changed approval.
prepared_silver["mapping_evidence"] = {
    **prepared_silver["mapping_evidence"],
    "approval_status": "accepted_with_limitations",
    "approval_decision_sha256": json_sha256(
        alabama_silver_acceptance
    ),
}

print("Alabama decision:", alabama_silver_acceptance["status"])
print("Eligible ozone hours:", decision_coverage["eligible_ozone_hours"])
print("Shared hours:", decision_coverage["shared_hours"])
print("Publication remains pending read-back validation.")
print("Decision report:", alabama_acceptance_path)

Alabama decision: accepted_with_limitations
Eligible ozone hours: 5071
Shared hours: 4836
Publication remains pending read-back validation.
Decision report: /home/ubuntu/dat535-final-project-data/dev/runs/0986e41d03894eabbfdfdacf001f3362/reports/alabama_silver_acceptance.json


### 8.4 Publish hourly Silver

The bundle contains the yearly mapping, hourly temperatures, ozone with temperature, hourly coverage, monthly coverage and gap intervals.

#### 8.4a Stage the six tables

We check that the acceptance decision matches this preparation and its source Silver release before writing the files.

In [37]:
# Clear results left over from an earlier staging or publication attempt.
hourly_publication = None
hourly_loaded_tables = None
hourly_validation = None
hourly_stage = None

# Require the prepared tables to belong to the selected source Silver publication.
if (
    prepared_silver["status"] != "prepared_unpublished"
    or prepared_silver["profile"] != hourly_manifest["profile"]
    or prepared_silver["source_silver_publication_id"]
    != hourly_manifest["publication_id"]
    or set(prepared_silver["tables"]) != set(hourly_output_keys)
):
    raise ValueError("Prepare hourly Silver from the selected source publication.")

if hourly_fixture_checks["status"] != "passed":
    raise ValueError("Run the hourly fixtures before staging.")

# Copy the profile, checks, approval and source details for this bundle.
hourly_evidence = json.loads(json.dumps({
    "profile": prepared_silver["profile"],
    "rules": prepared_silver["rules"],
    "mapping": prepared_silver["mapping_evidence"],
    "acceptance": alabama_silver_acceptance,
    "checks": prepared_silver["checks"],
    "fixtures": hourly_fixture_checks,
    # Identify the exact source Silver release used to prepare these tables.
    "source": {
        "publication_id": hourly_manifest["publication_id"],
        "manifest_path": Path(source_silver_publication["manifest_path"])
        .relative_to(context.environment_root).as_posix(),
        "manifest_sha256": source_silver_publication["manifest_sha256"],
        "rule_versions": hourly_manifest["rule_versions"],
    },
    "provenance": run_provenance,
}, allow_nan=False))
# Check that the approval and preparation evidence refer to the same mapping and source.
check_hourly_evidence(hourly_evidence)

# Define each hourly table's schema, key and expected count from the preparation evidence.
hourly_row_counts = hourly_expected_rows(hourly_evidence)
hourly_contracts = {
    name: {
        "schema_version": f"{name}-v1",
        "columns": dict(table.dtypes),
        "column_order": table.columns,
        "key": hourly_output_keys[name],
        "expected_rows": hourly_row_counts[name],
    }
    for name, table in prepared_silver["tables"].items()
}

# Reuse the Parquet staging helper to write the six hourly tables.
hourly_stage = stage_source_silver(
    prepared_silver["tables"], hourly_contracts, context,
)
print("Hourly status: staged, unpublished")

Staged yearly_mapping: 1 rows, 1 Parquet file(s).
Staged temperature_hourly: 8514 rows, 1 Parquet file(s).
Staged ozone_enriched: 5071 rows, 1 Parquet file(s).
Staged hourly_coverage: 8760 rows, 1 Parquet file(s).
Staged monthly_coverage: 12 rows, 1 Parquet file(s).
Staged hourly_gaps: 143 rows, 1 Parquet file(s).
Hourly status: staged, unpublished


#### 8.4b Compare the saved tables with the preparation

We read the Parquet files back and compare every column and row, including missing values, flags and source details. Matching row counts alone could miss a different weather report being selected.

In [38]:
# Clear any earlier validation result before checking the staged files.
hourly_validation = None
# Require the stage, contracts and acceptance evidence to describe the same preparation.
check_hourly_evidence(hourly_evidence)
if (
    hourly_stage["run_id"] != context.run_id
    or hourly_stage["environment"] != context.environment
    or hourly_stage["contract_sha256"] != json_sha256(hourly_contracts)
    or {name: item["expected_rows"] for name, item in hourly_contracts.items()}
    != hourly_expected_rows(hourly_evidence)
):
    raise ValueError("Stage, contracts and preparation evidence do not match.")

# The loader checks files and keys, then compares all columns and rows in both directions.
hourly_readback, hourly_dataset_checks = read_hourly_files(
    child_path(
        context.environment_root,
        hourly_stage["bundle_path"],
        must_exist=True,
    ),
    hourly_stage["datasets"],
    hourly_contracts,
    spark=spark,
    expected=prepared_silver["tables"],
)

# Record which staged files, contracts and evidence passed these comparisons.
# The publisher rejects this report if any of them have changed.
hourly_validation = {
    "status": "validated",
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": context.run_id,
    "stage_sha256": json_sha256(hourly_stage),
    "contract_sha256": json_sha256(hourly_contracts),
    "evidence_sha256": json_sha256(hourly_evidence),
    "datasets": hourly_dataset_checks,
}
# Save validation evidence before attempting publication.
write_json(
    context.reports_dir / "hourly_silver_validation.json",
    hourly_validation,
)
print("Hourly status: validated, unpublished")
print("Complete table comparisons passed:", len(hourly_dataset_checks))

Verified hourly_coverage: 8760 rows.
Verified hourly_gaps: 143 rows.
Verified monthly_coverage: 12 rows.
Verified ozone_enriched: 5071 rows.
Verified temperature_hourly: 8514 rows.
Verified yearly_mapping: 1 rows.
Hourly status: validated, unpublished
Complete table comparisons passed: 6


#### 8.4c Publish the validated bundle

The manifest is written after the file checks pass. Repeating the same publication attempt should return the same manifest. The cell below also checks that a failed validation blocks publication.

In [39]:
# Define publication for a validated hourly bundle, then publish the prepared bundle.
def publish_hourly_silver(stage, validation, contracts, evidence, context):
    """Publish the validated hourly tables and write the manifest after the files are in place."""
    # Copy the supplied metadata and reject values that cannot be recorded as standard JSON.
    stage, validation, contracts, evidence = json.loads(json.dumps(
        [stage, validation, contracts, evidence], allow_nan=False,
    ))
    # Check the reviewed mapping and source parent before any files are moved.
    check_hourly_evidence(evidence)
    
    # Calculate checksums so validation and publication refer to these exact inputs.
    stage_hash = json_sha256(stage)
    contract_hash = json_sha256(contracts)
    evidence_hash = json_sha256(evidence)
    root = Path(context.environment_root).resolve()

    # Require validation to match this run, staged bundle, contracts and evidence.
    if (
        stage["status"] != "staged"
        or stage["run_id"] != context.run_id
        or stage["environment"] != context.environment
        or Path(stage["environment_root"]).resolve() != root
        or stage["contract_sha256"] != contract_hash
        or validation["status"] != "validated"
        or validation["run_id"] != context.run_id
        or validation["stage_sha256"] != stage_hash
        or validation["contract_sha256"] != contract_hash
        or validation["evidence_sha256"] != evidence_hash
        or set(stage["datasets"]) != set(hourly_output_keys)
        or set(contracts) != set(hourly_output_keys)
        or set(validation["datasets"]) != set(hourly_output_keys)
        or {name: item["expected_rows"] for name, item in contracts.items()}
        != hourly_expected_rows(evidence)
    ):
        raise ValueError(
            "Publication requires matching stage, validation and evidence."
        )

    # Matching counts are not enough: every table must also match every prepared row and column.
    for name, info in stage["datasets"].items():
        check = validation["datasets"][name]
        if (
            check["status"] != "passed"
            or check["complete_values_match"] is not True
            or check["records"] != info["row_count"]
            or check["records"] != contracts[name]["expected_rows"]
            or check["parquet_files"] != len(info["files"])
        ):
            raise ValueError(
                f"{name}: complete read-back validation is required."
            )

    # Use the profile region and year in the release path.
    region = evidence["profile"]["region"].strip().lower().replace(" ", "_")
    year = evidence["profile"]["year"]
    # Keep the move within the selected environment and this run's staging directory.
    staged = child_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if (
        not staging_root.is_relative_to(root / "staging")
        or staged == staging_root
        or not staged.is_relative_to(staging_root)
    ):
        raise ValueError(
            "The bundle must be inside this run's staging directory."
        )

    # Identify the publication from both the staged bundle and its supporting evidence.
    publication_id = json_sha256({
        "stage_sha256": stage_hash,
        "evidence_sha256": evidence_hash,
    })
    suffix = f"silver/hourly/{region}/{year}/{publication_id}"
    final_bundle = child_path(root, f"data/{suffix}")
    manifest_path = child_path(root, f"published/{suffix}/manifest.json")
    publication_root = child_path(root, "published/silver/hourly")

    # Record the six datasets, their contracts, validation and parent-source evidence.
    identity = {
        "manifest_version": 1,
        "kind": "hourly_silver",
        "status": "published",
        "publication_id": publication_id,
        "run_id": context.run_id,
        "environment": context.environment,
        "bundle_path": final_bundle.relative_to(root).as_posix(),
        "stage_sha256": stage_hash,
        "contract_sha256": contract_hash,
        "evidence_sha256": evidence_hash,
        "contracts": contracts,
        "datasets": stage["datasets"],
        "evidence": evidence,
        "validation": validation,
        "storage": {
            key: stage[key]
            for key in (
                "format", "compression", "write_partitions",
                "partition_columns",
            )
        },
    }

    # Lock hourly publication so two runs cannot move files and write manifests at the same time.
    publication_root.mkdir(parents=True, exist_ok=True)
    lock = publication_root / ".publish.lock"
    try:
        lock.mkdir()
    except FileExistsError as error:
        raise RuntimeError(
            f"Publication lock already exists: {lock}"
        ) from error

    try:
        # Reuse an identical attempt only after its saved metadata and files have been checked.
        if manifest_path.exists():
            existing = json.loads(manifest_path.read_text(encoding="utf-8"))
            if {key: existing.get(key) for key in identity} != identity:
                raise ValueError(
                    "The existing publication has different metadata."
                )
            verify_file_inventory(
                final_bundle, existing["datasets"]
            )
            outcome, manifest = "reused", existing
        else:
            # An existing data directory without a manifest is incomplete and needs inspection.
            if final_bundle.exists() or final_bundle.is_symlink():
                raise FileExistsError(
                    f"An incomplete publication exists at {final_bundle}. "
                    "Inspect it, then stage a new attempt."
                )
            # Recheck the staged files before moving the bundle.
            verify_file_inventory(staged, stage["datasets"])
            final_bundle.parent.mkdir(parents=True, exist_ok=True)
            # Add the publication time to the metadata for this new release.
            manifest = {
                **identity,
                "published_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            # Move the data first. The manifest makes the complete release available to readers.
            staged.rename(final_bundle)
            write_json(manifest_path, manifest)
            outcome = "published"
        return {
            "outcome": outcome,
            "manifest_path": str(manifest_path),
            "manifest_sha256": file_sha256(manifest_path),
            "manifest": manifest,
        }
    # Always release the publication lock, including when an error is raised.
    finally:
        lock.rmdir()


# Failed validation must be rejected before publication.
try:
    publish_hourly_silver(
        hourly_stage,
        {**hourly_validation, "status": "failed"},
        hourly_contracts,
        hourly_evidence,
        context,
    )
except ValueError:
    print("Failed-validation guard: passed")
else:
    raise AssertionError("Failed validation was incorrectly accepted.")

# Publish the validated bundle with its source and acceptance evidence.
hourly_publication = publish_hourly_silver(
    hourly_stage, hourly_validation,
    hourly_contracts, hourly_evidence, context,
)
# The next cell reloads the published files.
hourly_readback = None

print("Publication outcome:", hourly_publication["outcome"])
print("Manifest:", hourly_publication["manifest_path"])
print("Manifest SHA-256:", hourly_publication["manifest_sha256"])

Failed-validation guard: passed
Publication outcome: published
Manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/hourly/alabama/2025/f2e02c38d9a6d46eadb1cb95aa33ea79ad7908319b5fed534cd68718f8e34179/manifest.json
Manifest SHA-256: 7ccf5850b0cf7a614280e2b1b870571ea27cbafef5a6097cfd953a1b19228e14


#### 8.4d Reload hourly Silver

The loader verifies the manifest, parent source release and six tables. The completion report records the manifest path and checksum for Gold.

Prepare stops after the following cell writes this report. Analyse opens the selected report in a fresh process; Full continues to Gold with the verified tables.


In [40]:
# Clear reload results before opening the hourly release.
hourly_loaded_manifest = None
hourly_loaded_tables = None
hourly_silver_completion = None

# Verify the hourly release, its source parent and its six tables.
(
    hourly_loaded_manifest,
    hourly_loaded_tables,
    hourly_load_checks,
) = load_hourly_silver(
    hourly_publication["manifest_path"],
    context,
    spark=spark,
    expected_manifest_sha256=hourly_publication["manifest_sha256"],
    profile=hourly_profile,
)

# Save the manifest identity and source parent for Full and Analyse.
hourly_silver_completion = {
    "status": "success",
    "readiness": "prepared",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": context.run_id,
    "environment": context.environment,
    "publication_id": hourly_loaded_manifest["publication_id"],
    "manifest_path": Path(hourly_publication["manifest_path"])
    .relative_to(context.environment_root).as_posix(),
    "manifest_sha256": hourly_publication["manifest_sha256"],
    "source_silver_publication_id": (
        hourly_loaded_manifest["evidence"]["source"]["publication_id"]
    ),
    "datasets": {
        name: check["records"]
        for name, check in hourly_load_checks.items()
    },
}
hourly_completion_path = (
    context.reports_dir / "hourly_silver_completion.json"
)
write_json(hourly_completion_path, hourly_silver_completion)

spark.createDataFrame(
    sorted(hourly_silver_completion["datasets"].items()),
    "dataset STRING, records LONG",
).show(truncate=False)

print("Hourly Silver status: prepared")
print("Published datasets reloaded:", len(hourly_loaded_tables))
print("Completion report:", hourly_completion_path)
print("Gold input manifest:", hourly_publication["manifest_path"])
print(
    "Gold input manifest SHA-256:",
    hourly_publication["manifest_sha256"],
)

Verified hourly_coverage: 8760 rows.
Verified hourly_gaps: 143 rows.
Verified monthly_coverage: 12 rows.
Verified ozone_enriched: 5071 rows.
Verified temperature_hourly: 8514 rows.
Verified yearly_mapping: 1 rows.
+------------------+-------+
|dataset           |records|
+------------------+-------+
|hourly_coverage   |8760   |
|hourly_gaps       |143    |
|monthly_coverage  |12     |
|ozone_enriched    |5071   |
|temperature_hourly|8514   |
|yearly_mapping    |1      |
+------------------+-------+

Hourly Silver status: prepared
Published datasets reloaded: 6
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/0986e41d03894eabbfdfdacf001f3362/reports/hourly_silver_completion.json
Gold input manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/hourly/alabama/2025/f2e02c38d9a6d46eadb1cb95aa33ea79ad7908319b5fed534cd68718f8e34179/manifest.json
Gold input manifest SHA-256: 7ccf5850b0cf7a614280e2b1b870571ea27cbafef5a6097cfd953a1b19228e14


## 9. Gold: sustained ozone periods in 2025

### 9.1 Rule and input

Silver keeps the observations and hourly coverage. Gold summarises them into one row per ozone period and a monthly table showing how much of the year was observed.

The primary rule, `gold-period-v1`, requires ozone **at or above 0.045 ppm for at least three consecutive UTC hours**. The threshold comes from the saved P90 calibration of eligible McAdory readings in `config/gold_rules.json`.

Missing, excluded or below-threshold ozone breaks a period. Missing temperature does not. Full uses the hourly Silver tables loaded above; Analyse opens a selected hourly completion report and can use another saved rule. The cell prints the actual rule and Silver input being analysed.


In [41]:
from gold import (
    build_monthly_coverage,
    build_period_catalogue,
    elevated_runs,
    load_gold_rules,
    summarise_year,
)
from gold_io import load_gold, publish_gold, write_gold_completion

# Load the saved Gold rule and its checksum.
rule_path = Path(os.environ.get("PIPELINE_GOLD_RULES", project_dir / "config/gold_rules.json"))
gold_rules = load_gold_rules(rule_path)
rule_config_sha256 = file_sha256(rule_path)

# Analyse selects a saved hourly receipt; Full uses this run's verified tables.
receipt_path = os.environ.get("PIPELINE_HOURLY_RECEIPT")

# Verify the receipt and hourly release, including the source parent.
if receipt_path:
    path = Path(receipt_path).resolve(strict=True)
    if path.name != "hourly_silver_completion.json" or not path.is_relative_to(
        context.environment_root / "runs"
    ):
        raise ValueError("Select an hourly Silver receipt from this environment.")

    receipt = json.loads(path.read_text(encoding="utf-8"))
    if receipt["status"] != "success" or receipt["environment"] != context.environment:
        raise ValueError("The selected hourly Silver run was not successful in this environment.")

    gold_parent_path = receipt["manifest_path"]
    gold_parent_sha = receipt["manifest_sha256"]
    gold_parent, gold_inputs, _ = load_hourly_silver(
        child_path(context.environment_root, gold_parent_path, must_exist=True),
        context,
        spark=spark,
        expected_manifest_sha256=gold_parent_sha,
        profile=alabama_profile,
    )
    if gold_parent["publication_id"] != receipt["publication_id"]:
        raise ValueError("The hourly Silver publication differs from the receipt.")
else:
    gold_parent = hourly_loaded_manifest
    gold_inputs = hourly_loaded_tables
    gold_parent_path = hourly_silver_completion["manifest_path"]
    gold_parent_sha = hourly_silver_completion["manifest_sha256"]

print("Rule:", gold_rules["rule_version"], gold_rules["comparison"],
      gold_rules["threshold_ppm"], "ppm for at least", gold_rules["minimum_hours"], "hours")
print("Hourly Silver:", gold_parent["environment"], gold_parent["publication_id"])

Rule: gold-period-v1 >= 0.045 ppm for at least 3 hours
Hourly Silver: dev f2e02c38d9a6d46eadb1cb95aa33ea79ad7908319b5fed534cd68718f8e34179


### 9.2a Find consecutive elevated hours

We order the elevated ozone readings by monitor and UTC hour. A gap of more than one hour starts a new run. We keep every hour in a run that meets the duration rule, including runs that cross midnight or the end of a month.

The detector compares timestamps, not just neighbouring row positions. For example, elevated readings at 10:00, 11:00 and 13:00 form a two-hour run and a one-hour run because 12:00 is missing from the elevated readings. Missing temperature does not break an ozone run.

In [42]:
# Cache the elevated-hour rows for the calculations below.
elevated = elevated_runs(gold_inputs["ozone_enriched"], gold_rules).cache()
# Keep every member hour of qualifying runs.
# A five-hour run contributes five hours to one period.
members = elevated.where(F.col("run_hours") >= gold_rules["minimum_hours"])
print("Elevated hours:", elevated.count())
print("Hours in qualifying periods:", members.count())

Elevated hours: 536
Hours in qualifying periods: 483


### 9.2b What happened during one period?

Each row in the catalogue describes one period. We report its duration, peak and mean ozone, and the temperatures available during those hours. The end time is the start of the first hour outside the period, whether or not that hour has an observation.

We also check the hour on either side. This shows whether we observed a lower ozone reading, had missing or excluded data, or reached the edge of the year.

For example, five member hours starting at 18:00, 19:00, 20:00, 21:00 and 22:00 form the interval from 18:00 up to, but not including, 23:00. Temperature averages use the available temperature values, and `temperature_hours` shows how many there were.

In [43]:
# Summarise each qualifying run and classify its boundaries.
catalogue = build_period_catalogue(
    members, gold_inputs["ozone_enriched"], gold_inputs["hourly_coverage"], gold_rules
)
catalogue.agg(
    F.count("*").alias("periods"),
    F.sum("duration_hours").alias("member_hours"),
).show()

# Show the earliest period and up to six of its member hours.
example = catalogue.orderBy("period_start_utc", "monitor_id").limit(1)
example.select(
    "period_start_utc", "period_end_utc", "duration_hours",
    "peak_ozone_ppm", "mean_ozone_ppm", "temperature_hours", "mean_temperature_c",
    "start_boundary", "end_boundary",
).show(truncate=False)

period_key = ["region", "year", "monitor_id", "period_start_utc"]
(
    members.join(example.select(*period_key), period_key)
    .select("hour_start_utc", "ozone_ppm", "temperature_c")
    .orderBy("hour_start_utc")
    .show(6, truncate=False)
)

+-------+------------+
|periods|member_hours|
+-------+------------+
|     74|         483|
+-------+------------+

+-------------------+-------------------+--------------+--------------+--------------------+-----------------+------------------+---------------------+---------------------+
|period_start_utc   |period_end_utc     |duration_hours|peak_ozone_ppm|mean_ozone_ppm      |temperature_hours|mean_temperature_c|start_boundary       |end_boundary         |
+-------------------+-------------------+--------------+--------------+--------------------+-----------------+------------------+---------------------+---------------------+
|2025-04-08 18:00:00|2025-04-08 23:00:00|5             |0.047         |0.045799999999999993|5                |17.56             |observed_non_elevated|observed_non_elevated|
+-------------------+-------------------+--------------+--------------+--------------------+-----------------+------------------+---------------------+---------------------+

+------------

### 9.3 Monthly observation coverage

We keep all twelve months from Silver so we can see how much ozone data was available, including months with no qualifying periods. We count each period in the month when it starts. If it crosses into another month, each hour counts in the month when it occurred.

In [44]:
# Count hours in their calendar month and periods in their starting month.
months = build_monthly_coverage(
    gold_inputs["monthly_coverage"], elevated, members, catalogue
)
months.select(
    "month_start_utc", "calendar_hours", "eligible_ozone_hours", "shared_hours",
    "elevated_hours", "qualifying_period_hours", "periods_started",
).orderBy("month_start_utc").show(12, truncate=False)

# Pool the annual counts before calculating percentages.
annual = summarise_year(months)
annual.select(
    "calendar_hours", "eligible_ozone_hours", "shared_hours",
    "ozone_hours_without_temperature", "temperature_match_pct",
    "elevated_hours", "qualifying_period_hours", "periods_started", "outcome",
).show(truncate=False)

+-------------------+--------------+--------------------+------------+--------------+-----------------------+---------------+
|month_start_utc    |calendar_hours|eligible_ozone_hours|shared_hours|elevated_hours|qualifying_period_hours|periods_started|
+-------------------+--------------+--------------------+------------+--------------+-----------------------+---------------+
|2025-01-01 00:00:00|744           |0                   |0           |0             |0                      |0              |
|2025-02-01 00:00:00|672           |0                   |0           |0             |0                      |0              |
|2025-03-01 00:00:00|744           |0                   |0           |0             |0                      |0              |
|2025-04-01 00:00:00|720           |684                 |684         |116           |115                    |12             |
|2025-05-01 00:00:00|744           |709                 |709         |57            |52                     |10       

### 9.4a How much does the duration rule matter?

We compare the selected rule's minimum duration with two hours, keeping the same Silver readings and cutoff. The primary rule uses three hours and 0.045 ppm. An alternative-rule Analyse run uses its selected cutoff for both durations.

This shows how many shorter periods the selected minimum leaves out. It does not change the rule or catalogue used for publication.

In [45]:
# Count each run once, including runs shorter than the selected minimum.
runs = elevated.where(F.col("hour_start_utc") == F.col("period_start_utc"))
# Compare durations on the same input and cutoff; keep the published catalogue.
for minimum in (gold_rules["minimum_hours"], 2):
    totals = runs.where(F.col("run_hours") >= minimum).agg(
        F.count("*").alias("periods"),
        F.coalesce(F.sum("run_hours"), F.lit(0)).alias("member_hours"),
    ).first()
    print(f"Minimum {minimum} hours: {totals.periods} periods, {totals.member_hours} member hours")

Minimum 3 hours: 74 periods, 483 member hours
Minimum 2 hours: 89 periods, 513 member hours


### 9.4b Publish and reload Gold

Save the period catalogue and monthly coverage together. The manifest lists their files, selected ozone rule and hourly Silver input.

Reloading checks the Gold files and schemas, and the checksum of the hourly Silver manifest. The completion report is written after reloading. The printed row counts come from the Gold manifest.


In [46]:
def show_gold_tables(tables):
    """Show the first 20 periods and all 12 months from loaded Gold tables."""
    print("\nOzone periods — timestamps are UTC; end times are exclusive:")
    (
        tables["period_catalogue"]
        .orderBy("period_start_utc")
        .select(
            "period_start_utc",
            "period_end_utc",
            "duration_hours",
            "peak_ozone_ppm",
            F.round("mean_ozone_ppm", 4).alias("mean_ozone_ppm"),
            F.round("mean_temperature_c", 1).alias("mean_temperature_c"),
            F.round("temperature_coverage_pct", 1).alias(
                "temperature_coverage_pct"
            ),
        )
        .show(20, truncate=False)
    )

    # Coverage helps distinguish no detected periods from missing observations.
    print("\nMonthly observation coverage and Gold results:")
    (
        tables["monthly_coverage"]
        .orderBy("month_start_utc")
        .select(
            F.date_format("month_start_utc", "yyyy-MM").alias("month"),
            "calendar_hours",
            F.col("eligible_ozone_hours").alias("ozone_hours"),
            F.col("eligible_temperature_hours").alias("temperature_hours"),
            "shared_hours",
            "elevated_hours",
            "periods_started",
            "qualifying_period_hours",
        )
        .show(12, truncate=False)
    )


# Save both tables with the selected rule and exact hourly Silver parent.
gold_release = publish_gold(
    catalogue=catalogue,
    monthly=months,
    rules=gold_rules,
    parent=gold_parent,
    parent_path=gold_parent_path,
    parent_sha256=gold_parent_sha,
    context=context,
    rule_config_sha256=rule_config_sha256,
)

# Verify and reopen the saved release before recording success.
released_manifest, released_tables = load_gold(
    gold_release["manifest_path"],
    expected_sha256=gold_release["manifest_sha256"],
    expected_parent_sha256=gold_parent_sha,
    context=context,
    spark=spark,
)

gold_completion_path = write_gold_completion(
    context, released_manifest, gold_release
)

# Free the cached working data; the saved Parquet files remain available.
elevated.unpersist()

print("Gold publication:", released_manifest["publication_id"])
print("Gold manifest SHA-256:", gold_release["manifest_sha256"])
print("Hourly Silver parent:", released_manifest["parent_hourly"]["publication_id"])
print(
    "Reloaded Gold tables:",
    {
        name: details["records"]
        for name, details in released_manifest["datasets"].items()
    },
)
print("Completion report:", gold_completion_path)

# Display this execution's verified results in both Full and Analyse.
show_gold_tables(released_tables)

Gold publication: 1987824683ce61d8d0c5e12bda19e648c6f89c7871c4b364534cdc52e61b1e9f
Gold manifest SHA-256: 39af7c53d9970d7f7c39f673bd6ae7a661c93da915abdd6b90ed7f6c62ae1144
Hourly Silver parent: f2e02c38d9a6d46eadb1cb95aa33ea79ad7908319b5fed534cd68718f8e34179
Reloaded Gold tables: {'period_catalogue': 74, 'monthly_coverage': 12}
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/0986e41d03894eabbfdfdacf001f3362/reports/gold_completion.json

Ozone periods — timestamps are UTC; end times are exclusive:
+-------------------+-------------------+--------------+--------------+--------------+------------------+------------------------+
|period_start_utc   |period_end_utc     |duration_hours|peak_ozone_ppm|mean_ozone_ppm|mean_temperature_c|temperature_coverage_pct|
+-------------------+-------------------+--------------+--------------+--------------+------------------+------------------------+
|2025-04-08 18:00:00|2025-04-08 23:00:00|5             |0.047         |0.0458        |1

## Summary

The pipeline uses PySpark to process five source files through Bronze, Silver and Gold. Bronze preserves the downloads and stores raw data as Parquet. Silver cleans and joins the records, while Gold produces the final summaries.

The published Gold tables are `period_catalogue`, containing 74 periods under the primary rule, and `monthly_coverage`, containing all 12 months, including those without ozone observations.

The saved tables and source details let us trace results back to the inputs. We can also reuse the prepared Silver data to run Gold with different rules without repeating ingestion and cleaning.

### View a previous production run

Call `show_production_gold` manually to display a completed production run. It checks the production completion report, published Gold files and the checksum of the saved hourly Silver manifest. The printed row counts come from the Gold manifest.

After defining the function, call it with a production run ID:

```python
show_production_gold("04699f2ea2ab4fedbce9117cbc39f7b5")
```


In [47]:
def show_production_gold(run_id):
    """Verify and display Gold from a selected completed production run."""
    root = Path("/home/ubuntu/dat535-final-project-data/prod")
    receipt_path = root / "runs" / run_id / "reports" / "gold_completion.json"
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))

    # Require a successful receipt from the selected production run.
    if (
        receipt["status"] != "success"
        or receipt["environment"] != "prod"
        or receipt["run_id"] != run_id
    ):
        raise ValueError(
            "The receipt does not identify the selected successful run."
        )

    # Describe existing storage without creating another run.
    context = RunContext(
        environment="prod",
        environment_root=root,
        run_id=run_id,
    )

    manifest, tables = load_gold(
        root / receipt["manifest_path"],
        expected_sha256=receipt["manifest_sha256"],
        expected_parent_sha256=receipt["parent_hourly"]["manifest_sha256"],
        context=context,
        spark=spark,
    )

    if manifest["publication_id"] != receipt["publication_id"]:
        raise ValueError("The loaded publication differs from the receipt.")

    print("Production run:", run_id)
    print("Gold publication:", manifest["publication_id"])
    print(
        "Verified table counts:",
        {
            name: details["records"]
            for name, details in manifest["datasets"].items()
        },
    )
    show_gold_tables(tables)